In [1]:
# Install required libraries
!pip install fastdtw scipy -q

# Now import them
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean

print("FastDTW installed successfully!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
FastDTW installed successfully!


In [2]:
import sys

print(sys.executable)

/usr/bin/python3


# Data Preprocessing

In [3]:
# ============================================================
# SMARTFALLMM FINAL PREPROCESSING PIPELINE
# LIGHTHART + KD READY
# FINAL CLEAN VERSION
# ============================================================

import os
import glob
import re
from collections import defaultdict

import numpy as np
import pandas as pd

from scipy.signal import (
    butter,
    filtfilt,
    find_peaks,
)

from fastdtw import fastdtw

from scipy.spatial.distance import euclidean

# ============================================================
# CONFIG
# ============================================================

DATA_PATH = r"/kaggle/input/datasets/pratyaydhar/smartfallmmdataset/SmartFallMM-Dataset-main"

BASE = DATA_PATH

SAVE_PATH = r"/kaggle/working/smartfallmm_lighthart_final.npz"

WINDOW     = 64
OVERLAP    = 32
GAP        = 20

NUM_JOINTS = 32
NUM_COORDS = 3

# ============================================================
# BUTTERWORTH FILTER
# ============================================================

def butterworth_filter(
    data,
    cutoff=7.5,
    fs=25,
    order=4
):

    min_samples = 3 * order + 1

    if len(data) < min_samples:
        return data

    nyquist = 0.5 * fs

    normal_cutoff = cutoff / nyquist

    b, a = butter(
        order,
        normal_cutoff,
        btype="low"
    )

    return filtfilt(
        b,
        a,
        data,
        axis=0
    )

# ============================================================
# CSV LOADER
# ============================================================

def csv_loader(path):

    try:

        df = pd.read_csv(
            path,
            header=0
        )

        if df.shape[1] == 0:
            return None

        df = df.dropna()

        df = df.bfill()

        df = df.reset_index(drop=True)

        return df if len(df) > 0 else None

    except Exception as e:

        print(f"[ERROR] {path}")

        print(e)

        return None

# ============================================================
# SMV COMPUTATION
# ============================================================

def compute_smv(acc_xyz):

    mean = np.mean(
        acc_xyz,
        axis=0,
        keepdims=True
    )

    zero_mean = acc_xyz - mean

    smv = np.sqrt(
        np.sum(
            zero_mean ** 2,
            axis=1,
            keepdims=True
        )
    )

    return smv

# ============================================================
# DTW HELPERS
# ============================================================

def filter_repeated_ids(path):

    seen_a = set()

    seen_s = set()

    acc_ids = []

    skl_ids = []

    for a, s in path:

        if a not in seen_a and s not in seen_s:

            seen_a.add(a)

            seen_s.add(s)

            acc_ids.append(a)

            skl_ids.append(s)

    return acc_ids, skl_ids

# ============================================================
# DTW ALIGNMENT
# ============================================================

def align_dtw(
    acc_filtered,
    acc_norm,
    skl,
    gap_mask
):

    wrist = skl[:, 9, :]

    acc_frob = np.linalg.norm(
        acc_filtered,
        axis=1
    )

    wrist_frob = np.linalg.norm(
        wrist,
        axis=1
    )

    _, path = fastdtw(

        acc_frob[:, np.newaxis],

        wrist_frob[:, np.newaxis],

        dist=euclidean
    )

    acc_ids, skl_ids = filter_repeated_ids(path)

    return (

        acc_norm[acc_ids],

        skl[skl_ids],

        gap_mask[acc_ids]
    )

# ============================================================
# LOAD SINGLE TRIAL
# ============================================================

def load_trial(
    acc_path,
    skl_path
):

    # --------------------------------------------------------
    # ACCELEROMETER
    # --------------------------------------------------------

    acc_df = csv_loader(acc_path)

    if acc_df is None:
        return None, None

    acc_raw = acc_df.iloc[
        2:,
        -3:
    ].to_numpy(dtype=np.float32)

    if len(acc_raw) < WINDOW:
        return None, None

    # --------------------------------------------------------
    # SKELETON
    # --------------------------------------------------------

    skl_df = csv_loader(skl_path)

    if skl_df is None:
        return None, None

    skl_raw = skl_df.iloc[
        :,
        -96:
    ].to_numpy(dtype=np.float32)

    if skl_raw.shape[1] != 96:
        return None, None

    skl_raw = skl_raw.reshape(
        -1,
        NUM_JOINTS,
        NUM_COORDS
    )

    if len(skl_raw) < WINDOW:
        return None, None

    return acc_raw, skl_raw

# ============================================================
# MERGE MULTIPLE TRIALS
# ============================================================

def merge_trials(trial_list):

    trial_list = sorted(
        trial_list,
        key=lambda x: x[0]
    )

    acc_parts = []

    skl_parts = []

    gap_parts = []

    acc_gap = np.zeros(
        (GAP, 3),
        dtype=np.float32
    )

    skl_gap = np.zeros(
        (GAP, NUM_JOINTS, NUM_COORDS),
        dtype=np.float32
    )

    for i, (_, acc_raw, skl_raw) in enumerate(trial_list):

        acc_parts.append(acc_raw)

        skl_parts.append(skl_raw)

        gap_parts.append(

            np.zeros(
                len(acc_raw),
                dtype=bool
            )
        )

        if i < len(trial_list) - 1:

            acc_parts.append(acc_gap)

            skl_parts.append(skl_gap)

            gap_parts.append(

                np.ones(
                    GAP,
                    dtype=bool
                )
            )

    merged_acc = np.concatenate(acc_parts)

    merged_skl = np.concatenate(skl_parts)

    gap_mask = np.concatenate(gap_parts)

    return merged_acc, merged_skl, gap_mask

# ============================================================
# WINDOW EXTRACTION
# ============================================================

def extract_windows_with_gap_mask(
    acc,
    skl,
    label,
    gap_mask
):

    windows = []

    seen_ranges = set()

    magnitude = acc[:, 3]

    height = 1.4 if label == 1 else 1.2

    distance = 50 if label == 1 else 100

    peaks, _ = find_peaks(

        magnitude,

        height=height,

        distance=distance
    )

    def valid_window(start, end):

        if start < 0 or end > len(acc):
            return False

        if gap_mask[start:end].any():
            return False

        if (start, end) in seen_ranges:
            return False

        return True

    # --------------------------------------------------------
    # PEAK WINDOWS
    # --------------------------------------------------------

    for p in peaks:

        start = max(
            0,
            p - WINDOW // 2
        )

        end = start + WINDOW

        if end > len(acc):

            end = len(acc)

            start = end - WINDOW

        if not valid_window(start, end):
            continue

        seen_ranges.add((start, end))

        windows.append(

            (
                acc[start:end],
                skl[start:end]
            )
        )

    # --------------------------------------------------------
    # SLIDING WINDOWS
    # --------------------------------------------------------

    for start in range(
        0,
        len(acc) - WINDOW + 1,
        OVERLAP
    ):

        end = start + WINDOW

        if not valid_window(start, end):
            continue

        seen_ranges.add((start, end))

        windows.append(

            (
                acc[start:end],
                skl[start:end]
            )
        )

    return windows

# ============================================================
# PROCESS GROUP
# ============================================================

def process_group(
    trial_list,
    label
):

    if not trial_list:
        return []

    merged_acc, merged_skl, gap_mask = merge_trials(
        trial_list
    )

    if len(merged_acc) < WINDOW:
        return []

    # --------------------------------------------------------
    # FILTER
    # --------------------------------------------------------

    acc_filtered = butterworth_filter(
        merged_acc
    )

    # --------------------------------------------------------
    # SMV
    # --------------------------------------------------------

    smv = compute_smv(
        acc_filtered
    )

    acc_with_smv = np.concatenate(
        [acc_filtered, smv],
        axis=1
    )

    # --------------------------------------------------------
    # NORMALIZATION
    # --------------------------------------------------------

    acc_norm = (

        acc_with_smv -

        acc_with_smv.mean(axis=0)

    ) / (

        acc_with_smv.std(axis=0) + 1e-6
    )

    # --------------------------------------------------------
    # HIP CENTERING
    # --------------------------------------------------------

    merged_skl = (

        merged_skl -

        merged_skl[:, 0:1, :]
    )

    # --------------------------------------------------------
    # DTW
    # --------------------------------------------------------

    acc_norm, merged_skl, gap_mask = align_dtw(

        acc_filtered,

        acc_norm,

        merged_skl,

        gap_mask
    )

    length = min(

        len(acc_norm),

        len(merged_skl),

        len(gap_mask)
    )

    acc_norm = acc_norm[:length]

    merged_skl = merged_skl[:length]

    gap_mask = gap_mask[:length]

    if length < WINDOW:
        return []

    return extract_windows_with_gap_mask(

        acc_norm,

        merged_skl,

        label,

        gap_mask
    )

# ============================================================
# BUILD DATASET
# ============================================================

def build_dataset():

    acc_set = []

    skl_set = []

    labels = []

    subjects = []

    activities = []

    skeleton_files = (

        glob.glob(f"{BASE}/young/skeleton/*.csv")

        +

        glob.glob(f"{BASE}/old/skeleton/*.csv")
    )

    print(f"\nSkeleton Files Found: {len(skeleton_files)}")

    group_map = defaultdict(list)

    for skel_file in skeleton_files:

        filename = os.path.basename(skel_file)

        match = re.findall(

            r"S\d+A\d+T\d+",

            filename
        )

        if not match:
            continue

        desc = match[0]

        subject = int(
            re.search(r"S(\d+)", desc).group(1)
        )

        act_num = int(
            re.search(r"A(\d+)", desc).group(1)
        )

        trial_num = int(
            re.search(r"T(\d+)", desc).group(1)
        )

        group = (

            "young"

            if "young" in skel_file

            else "old"
        )

        acc_file = (

            f"{BASE}/{group}/accelerometer/watch/{desc}.csv"
        )

        if not os.path.exists(acc_file):
            continue

        group_map[(subject, act_num)].append(

            (
                trial_num,
                acc_file,
                skel_file
            )
        )

    print(f"\nUnique Groups: {len(group_map)}")

    total_windows = 0

    valid_groups = 0

    for (subject, act_num), file_entries in sorted(group_map.items()):

        label = int(act_num > 9)

        trial_data = []

        for trial_num, acc_path, skl_path in file_entries:

            acc_raw, skl_raw = load_trial(

                acc_path,

                skl_path
            )

            if acc_raw is None:
                continue

            trial_data.append(

                (
                    trial_num,
                    acc_raw,
                    skl_raw
                )
            )

        if not trial_data:
            continue

        samples = process_group(
            trial_data,
            label
        )

        if not samples:
            continue

        valid_groups += 1

        total_windows += len(samples)

        for acc_w, skl_w in samples:

            acc_set.append(acc_w)

            skl_set.append(skl_w)

            labels.append(label)

            subjects.append(subject)

            activities.append(act_num)

    # ========================================================
    # FINAL ARRAYS
    # ========================================================

    acc_set = np.array(
        acc_set,
        dtype=np.float32
    )

    skl_set = np.array(
        skl_set,
        dtype=np.float32
    )

    labels = np.array(
        labels,
        dtype=np.int32
    )

    subjects = np.array(
        subjects,
        dtype=np.int32
    )

    activities = np.array(
        activities,
        dtype=np.int32
    )

    print("\n" + "=" * 60)

    print("FINAL DATASET")

    print("=" * 60)

    print(f"Valid Groups      : {valid_groups}")

    print(f"Total Windows     : {total_windows}")

    print(f"Accelerometer     : {acc_set.shape}")

    print(f"Skeleton          : {skl_set.shape}")

    print(f"Labels            : {labels.shape}")

    print(f"Subjects          : {subjects.shape}")

    print(f"Activities        : {activities.shape}")

    print("=" * 60)

    return {

        "accelerometer": acc_set,

        "skeleton": skl_set,

        "labels": labels,

        "subjects": subjects,

        "activities": activities,
    }

# ============================================================
# BUILD DATASET
# ============================================================

dataset = build_dataset()

# ============================================================
# SAVE FINAL DATASET
# ============================================================

np.savez(

    SAVE_PATH,

    accelerometer = dataset["accelerometer"],

    skeleton = dataset["skeleton"],

    labels = dataset["labels"],

    subjects = dataset["subjects"],

    activities = dataset["activities"],
)

print("\n✅ FINAL DATASET SAVED")

print(f"\nSaved To:\n{SAVE_PATH}")

# ============================================================
# VERIFY SAVED DATASET
# ============================================================

saved = np.load(

    SAVE_PATH,

    allow_pickle=True,
)

print("\nSaved Keys:\n")

print(saved.files)

print("\nSaved Accelerometer Shape:")

print(saved["accelerometer"].shape)

print("\nSaved Skeleton Shape:")

print(saved["skeleton"].shape)

print("\n✅ PREPROCESSING COMPLETE")


Skeleton Files Found: 2389

Unique Groups: 464
[ERROR] /kaggle/input/datasets/pratyaydhar/smartfallmmdataset/SmartFallMM-Dataset-main/young/skeleton/S48A02T05.csv
No columns to parse from file
[ERROR] /kaggle/input/datasets/pratyaydhar/smartfallmmdataset/SmartFallMM-Dataset-main/young/skeleton/S50A01T03.csv
No columns to parse from file
[ERROR] /kaggle/input/datasets/pratyaydhar/smartfallmmdataset/SmartFallMM-Dataset-main/young/skeleton/S50A01T02.csv
No columns to parse from file
[ERROR] /kaggle/input/datasets/pratyaydhar/smartfallmmdataset/SmartFallMM-Dataset-main/young/skeleton/S50A02T04.csv
No columns to parse from file
[ERROR] /kaggle/input/datasets/pratyaydhar/smartfallmmdataset/SmartFallMM-Dataset-main/young/skeleton/S50A02T05.csv
No columns to parse from file
[ERROR] /kaggle/input/datasets/pratyaydhar/smartfallmmdataset/SmartFallMM-Dataset-main/young/skeleton/S50A05T02.csv
No columns to parse from file
[ERROR] /kaggle/input/datasets/pratyaydhar/smartfallmmdataset/SmartFallMM-Da

# Data Loaders

In [4]:
# ============================================================
# SMARTFALLMM FINAL LIGHTHART PIPELINE
# REPO-ALIGNED STATIC SPLIT
# FINAL CLEAN VERSION
# ============================================================

import numpy as np
import torch

from torch.utils.data import (
    Dataset,
    DataLoader,
)

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)

print(f"\nUsing Device: {DEVICE}")

# ============================================================
# LOAD FINAL PREPROCESSED DATASET
# ============================================================

DATASET_PATH = \
r"/kaggle/working/smartfallmm_lighthart_final.npz"

dataset = np.load(

    DATASET_PATH,

    allow_pickle=True,
)

print("\n✅ FINAL DATASET LOADED")

print("\nDataset Keys:\n")

print(dataset.files)

# ============================================================
# LOAD ARRAYS
# ============================================================

accelerometer = dataset["accelerometer"]

skeleton = dataset["skeleton"]

labels = dataset["labels"]

subjects = dataset["subjects"]

activities = dataset["activities"]

# ============================================================
# DATASET SHAPES
# ============================================================

print("\n" + "=" * 60)

print("DATASET SHAPES")

print("=" * 60)

print(

    f"Accelerometer : {accelerometer.shape}"
)

print(

    f"Skeleton      : {skeleton.shape}"
)

print(

    f"Labels        : {labels.shape}"
)

print(

    f"Subjects      : {subjects.shape}"
)

print(

    f"Activities    : {activities.shape}"
)

print("=" * 60)

# ============================================================
# LABEL DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)

print("LABEL DISTRIBUTION")

print("=" * 60)

unique, counts = np.unique(

    labels,

    return_counts=True
)

print(

    dict(zip(unique, counts))
)

print("=" * 60)

# ============================================================
# SUBJECT SAMPLE COUNTS
# ============================================================

print("\n" + "=" * 60)

print("SUBJECT SAMPLE COUNTS")

print("=" * 60)

unique_subjects = sorted(
    np.unique(subjects)
)

subject_sample_count = {}

for s in unique_subjects:

    count = int(np.sum(subjects == s))

    subject_sample_count[s] = count

    print(f"S{s}: {count}")

print("=" * 60)

# ============================================================
# FINAL REPO-ALIGNED STATIC SPLIT
# ============================================================

# ------------------------------------------------------------
# TRAIN:
# Young multimodal subjects + selected old ADL subjects
#
# VALIDATION:
# KEEP SAME AS REPO
#
# TEST:
# Unseen young fall-capable subjects
# ------------------------------------------------------------

train_subjects = [

    # Young subjects
    29, 30, 31, 32,
    34, 35, 36, 37,
    39, 43, 44, 45,
    48, 49, 50, 51,
    52, 53, 54, 55,

    # Selected old subjects
    15, 16, 17, 18,
    19, 21
]

# ------------------------------------------------------------
# KEEP SAME AS REPO
# ------------------------------------------------------------

val_subjects = [

    38, 46
]

# ------------------------------------------------------------
# HARD UNSEEN TEST SUBJECTS
# ------------------------------------------------------------

test_subjects = [

    56, 57, 58, 59,
    60, 61, 62, 63
]

# ============================================================
# CREATE SUBSET
# ============================================================

def create_subset(subject_ids):

    mask = np.isin(

        subjects,

        subject_ids
    )

    return {

        "accelerometer":

            accelerometer[mask],

        "skeleton":

            skeleton[mask],

        "labels":

            labels[mask],

        "subjects":

            subjects[mask],

        "activities":

            activities[mask],
    }

# ============================================================
# CREATE SPLITS
# ============================================================

train_data = create_subset(
    train_subjects
)

val_data = create_subset(
    val_subjects
)

test_data = create_subset(
    test_subjects
)

# ============================================================
# STRICT LEAKAGE CHECK
# ============================================================

print("\n" + "=" * 60)

print("STRICT SUBJECT LEAKAGE CHECK")

print("=" * 60)

train_subs = set(
    np.unique(train_data["subjects"])
)

val_subs = set(
    np.unique(val_data["subjects"])
)

test_subs = set(
    np.unique(test_data["subjects"])
)

print(

    f"Train Subjects : "
    f"{sorted(list(train_subs))}"
)

print(

    f"Val Subjects   : "
    f"{sorted(list(val_subs))}"
)

print(

    f"Test Subjects  : "
    f"{sorted(list(test_subs))}"
)

print("-" * 60)

print(

    f"Train ∩ Val  : "
    f"{train_subs.intersection(val_subs)}"
)

print(

    f"Train ∩ Test : "
    f"{train_subs.intersection(test_subs)}"
)

print(

    f"Val ∩ Test   : "
    f"{val_subs.intersection(test_subs)}"
)

assert len(

    train_subs.intersection(val_subs)

) == 0

assert len(

    train_subs.intersection(test_subs)

) == 0

assert len(

    val_subs.intersection(test_subs)

) == 0

print("\n✅ PASS: No Subject Leakage")

# ============================================================
# CLASS DISTRIBUTION
# ============================================================

def print_stats(name, data):

    y = data["labels"]

    falls = int(np.sum(y == 1))

    adls = int(np.sum(y == 0))

    total = len(y)

    ratio = adls / (falls + 1e-6)

    print(

        f"{name:<6} | "

        f"Total={total:<6} | "

        f"Falls={falls:<6} | "

        f"ADLs={adls:<6} | "

        f"ADL/Fall Ratio={ratio:.2f}"
    )

print("\n" + "=" * 60)

print("CLASS DISTRIBUTION")

print("=" * 60)

print_stats("TRAIN", train_data)

print_stats("VAL", val_data)

print_stats("TEST", test_data)

print("=" * 60)

# ============================================================
# DATASET CLASS
# ============================================================

class SmartFallFeeder(Dataset):

    def __init__(self, data):

        self.acc = torch.tensor(

            data["accelerometer"],

            dtype=torch.float32,
        )

        self.skl = torch.tensor(

            data["skeleton"],

            dtype=torch.float32,
        )

        self.labels = torch.tensor(

            data["labels"],

            dtype=torch.float32,
        )

        print(

            f"Loaded {len(self.labels)} samples"
        )

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        acc = self.acc[idx]

        skl = self.skl[idx]

        label = self.labels[idx].unsqueeze(0)

        data = {

            "accelerometer": acc,

            "skeleton": skl,
        }

        return data, label

# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = SmartFallFeeder(
    train_data
)

val_dataset = SmartFallFeeder(
    val_data
)

test_dataset = SmartFallFeeder(
    test_data
)

# ============================================================
# DATALOADERS
# ============================================================

BATCH_SIZE = 32

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=True,

    drop_last=True,
)

val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=True,
)

test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=True,
)

# ============================================================
# FINAL PIPELINE CHECK
# ============================================================

print("\n" + "=" * 60)

print("PIPELINE READY")

print("=" * 60)

print(

    f"Train Samples : "
    f"{len(train_dataset)}"
)

print(

    f"Val Samples   : "
    f"{len(val_dataset)}"
)

print(

    f"Test Samples  : "
    f"{len(test_dataset)}"
)

print("-" * 60)

print(

    f"Train Batches : "
    f"{len(train_loader)}"
)

print(

    f"Val Batches   : "
    f"{len(val_loader)}"
)

print(

    f"Test Batches  : "
    f"{len(test_loader)}"
)

print("=" * 60)

# ============================================================
# SANITY CHECK
# ============================================================

sample_data, sample_label = next(
    iter(train_loader)
)

print("\n" + "=" * 60)

print("SANITY CHECK")

print("=" * 60)

print(

    "Accelerometer Shape :",

    sample_data["accelerometer"].shape
)

print(

    "Skeleton Shape      :",

    sample_data["skeleton"].shape
)

print(

    "Labels Shape        :",

    sample_label.shape
)

print("\n✅ EVERYTHING READY")


Using Device: cuda

✅ FINAL DATASET LOADED

Dataset Keys:

['accelerometer', 'skeleton', 'labels', 'subjects', 'activities']

DATASET SHAPES
Accelerometer : (13077, 64, 4)
Skeleton      : (13077, 64, 32, 3)
Labels        : (13077,)
Subjects      : (13077,)
Activities    : (13077,)

LABEL DISTRIBUTION
{np.int32(0): np.int64(10330), np.int32(1): np.int64(2747)}

SUBJECT SAMPLE COUNTS
S2: 165
S3: 181
S4: 101
S5: 139
S6: 152
S7: 130
S8: 179
S9: 139
S10: 172
S11: 198
S13: 20
S14: 178
S15: 245
S16: 225
S17: 274
S18: 193
S19: 263
S21: 188
S22: 190
S23: 226
S25: 16
S26: 70
S29: 132
S30: 205
S31: 212
S32: 201
S34: 117
S35: 174
S36: 79
S37: 228
S38: 323
S39: 294
S43: 311
S44: 718
S45: 219
S46: 619
S47: 433
S48: 345
S49: 358
S50: 122
S51: 208
S52: 383
S53: 364
S54: 383
S55: 433
S56: 349
S57: 380
S58: 321
S59: 111
S60: 294
S61: 285
S62: 413
S63: 419

STRICT SUBJECT LEAKAGE CHECK
Train Subjects : [np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(21), np.int32(29), np.

In [5]:
print(train_loader)

print(val_loader)

print(test_loader)

# IT Baseline

In [6]:
# ============================================================
# FINAL IMPROVED IT STUDENT
# CNN + TRANSFORMER HYBRID
# KD-READY
# SMARTFALLMM
# ============================================================

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"\nUsing Device: {DEVICE}")

# ============================================================
# DROPPATH
# ============================================================

class DropPath(nn.Module):

    def __init__(self, drop_prob=0.0):

        super().__init__()

        self.drop_prob = drop_prob

    def forward(self, x):

        if (
            self.drop_prob == 0.0
            or
            not self.training
        ):
            return x

        keep_prob = 1 - self.drop_prob

        shape = (
            x.shape[0],
        ) + (1,) * (x.ndim - 1)

        random_tensor = torch.empty(

            shape,

            dtype=x.dtype,

            device=x.device,

        ).bernoulli_(keep_prob)

        return x * random_tensor.div(keep_prob)

# ============================================================
# TEMPORAL ATTENTION POOLING
# ============================================================

class TemporalAttentionPooling(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.attn = nn.Sequential(

            nn.LayerNorm(dim),

            nn.Linear(dim, dim // 2),

            nn.GELU(),

            nn.Dropout(0.1),

            nn.Linear(dim // 2, 1),
        )

    def forward(self, x):

        weights = self.attn(x)

        weights = torch.softmax(
            weights,
            dim=1,
        )

        pooled = (x * weights).sum(dim=1)

        return pooled

# ============================================================
# TRANSFORMER BLOCK
# ============================================================

class ITBlock(nn.Module):

    def __init__(

        self,

        dim,

        heads,

        mlp_ratio=4,

        dropout=0.1,

        drop_path=0.1,
    ):

        super().__init__()

        self.norm1 = nn.LayerNorm(dim)

        self.attn = nn.MultiheadAttention(

            dim,

            heads,

            dropout=dropout,

            batch_first=True,
        )

        self.drop_path = DropPath(drop_path)

        self.norm2 = nn.LayerNorm(dim)

        hidden_dim = int(dim * mlp_ratio)

        self.mlp = nn.Sequential(

            nn.Linear(dim, hidden_dim),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(hidden_dim, dim),

            nn.Dropout(dropout),
        )

    def forward(self, x):

        residual = x

        x = self.norm1(x)

        x, _ = self.attn(

            x,
            x,
            x,

            need_weights=False
        )

        x = residual + self.drop_path(x)

        residual = x

        x = self.norm2(x)

        x = self.mlp(x)

        x = residual + self.drop_path(x)

        return x

# ============================================================
# FINAL STUDENT MODEL
# ============================================================

class FinalITStudent(nn.Module):

    def __init__(

        self,

        frames=64,

        channels=4,

        embed_dim=128,

        depth=5,

        heads=8,

        dropout=0.1,

        drop_path=0.15,
    ):

        super().__init__()

        self.frames = frames

        self.embed_dim = embed_dim

        # ====================================================
        # CNN STEM
        # ====================================================

        self.conv_stem = nn.Sequential(

            nn.Conv1d(
                channels,
                32,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm1d(32),

            nn.GELU(),

            nn.Conv1d(
                32,
                64,
                kernel_size=5,
                padding=2,
            ),

            nn.BatchNorm1d(64),

            nn.GELU(),

            nn.Conv1d(
                64,
                128,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm1d(128),

            nn.GELU(),
        )

        # ====================================================
        # PATCH EMBEDDING
        # ====================================================

        self.patch_embed = nn.Sequential(

            nn.Linear(
                4 * 128,
                embed_dim,
            ),

            nn.LayerNorm(embed_dim),
        )

        # ====================================================
        # TOKENS
        # ====================================================

        self.cls_token = nn.Parameter(

            torch.zeros(
                1,
                1,
                embed_dim,
            )
        )

        num_tokens = (
            frames // 4
        ) + 1

        self.pos_embed = nn.Parameter(

            torch.zeros(
                1,
                num_tokens,
                embed_dim,
            )
        )

        self.pos_drop = nn.Dropout(dropout)

        # ====================================================
        # STOCHASTIC DEPTH
        # ====================================================

        dpr = [

            x.item()

            for x in torch.linspace(
                0,
                drop_path,
                depth,
            )
        ]

        # ====================================================
        # TRANSFORMER BLOCKS
        # ====================================================

        self.blocks = nn.ModuleList([

            ITBlock(

                dim=embed_dim,

                heads=heads,

                dropout=dropout,

                drop_path=dpr[i],
            )

            for i in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)

        self.pool = TemporalAttentionPooling(
            embed_dim
        )

        # ====================================================
        # FEATURE HEAD
        # ====================================================

        self.feature_head = nn.Sequential(

            nn.LayerNorm(embed_dim * 2),

            nn.Dropout(0.2),
        )

        # ====================================================
        # KD PROJECTOR
        # ====================================================

        self.kd_projector = nn.Sequential(

            nn.Linear(
                embed_dim * 2,
                embed_dim,
            ),

            nn.LayerNorm(embed_dim),
        )

        # ====================================================
        # CLASSIFIER
        # ====================================================

        self.classifier = nn.Sequential(

            nn.Linear(
                embed_dim * 2,
                embed_dim,
            ),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                embed_dim,
                1,
            ),
        )

        # ====================================================
        # INIT
        # ====================================================

        nn.init.trunc_normal_(
            self.cls_token,
            std=0.02,
        )

        nn.init.trunc_normal_(
            self.pos_embed,
            std=0.02,
        )

    # ========================================================
    # PATCH CREATION
    # ========================================================

    def create_patches(self, x):

        B, T, C = x.shape

        x = x.reshape(

            B,

            T // 4,

            4 * C,
        )

        x = self.patch_embed(x)

        return x

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(self, data):

        x = data["accelerometer"]

        B = x.shape[0]

        # ====================================================
        # GAUSSIAN NOISE AUGMENTATION
        # ====================================================

        if self.training:

            noise = torch.randn_like(x) * 0.01

            x = x + noise

        # ====================================================
        # CNN STEM
        # ====================================================

        x = x.permute(0, 2, 1)

        x = self.conv_stem(x)

        x = x.permute(0, 2, 1)

        # ====================================================
        # PATCH TOKENS
        # ====================================================

        x = self.create_patches(x)

        # ====================================================
        # CLS TOKEN
        # ====================================================

        cls = self.cls_token.expand(

            B,
            -1,
            -1,
        )

        x = torch.cat(
            [cls, x],
            dim=1,
        )

        # ====================================================
        # POSITIONAL EMBEDDINGS
        # ====================================================

        x = x + self.pos_embed

        x = self.pos_drop(x)

        # ====================================================
        # TRANSFORMER
        # ====================================================

        for blk in self.blocks:

            x = blk(x)

        x = self.norm(x)

        # ====================================================
        # CLS + ATTENTION POOL
        # ====================================================

        cls_token = x[:, 0]

        pooled = self.pool(x[:, 1:])

        # ====================================================
        # RAW FEATURES
        # ====================================================

        raw_features = torch.cat(

            [
                cls_token,
                pooled,
            ],

            dim=1,
        )

        raw_features = self.feature_head(
            raw_features
        )

        # ====================================================
        # KD FEATURES
        # ====================================================

        kd_features = self.kd_projector(
            raw_features
        )

        kd_features = F.normalize(

            kd_features,

            dim=-1,
        )

        # ====================================================
        # CLASSIFICATION
        # ====================================================

        logits = self.classifier(
            raw_features
        )

        return logits, kd_features, raw_features

# ============================================================
# FIND BEST THRESHOLD
# ============================================================

@torch.no_grad()
def find_best_threshold(

    model,

    loader,
):

    model.eval()

    all_probs = []

    all_labels = []

    for data, labels in loader:

        data = {

            k: v.to(DEVICE)

            for k, v in data.items()
        }

        logits, _, _ = model(data)

        probs = torch.sigmoid(
            logits.view(-1)
        )

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

    best_acc = 0.0

    best_threshold = 0.5

    thresholds = np.arange(
        0.30,
        0.81,
        0.02
    )

    for t in thresholds:

        preds = (
            np.array(all_probs) >= t
        ).astype(int)

        acc = accuracy_score(
            all_labels,
            preds,
        )

        if acc > best_acc:

            best_acc = acc

            best_threshold = t

    print(
        f"\nBest Threshold: {best_threshold:.2f}"
    )

    print(
        f"Best Validation Accuracy: {best_acc:.4f}"
    )

    return best_threshold

# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(

    model,

    loader,

    criterion,

    threshold=0.5,
):

    model.eval()

    total_loss = 0.0

    all_probs = []

    all_preds = []

    all_labels = []

    for data, labels in loader:

        data = {

            k: v.to(DEVICE)

            for k, v in data.items()
        }

        labels = labels.float().to(
            DEVICE
        ).view(-1)

        logits, _, _ = model(data)

        logits = logits.view(-1)

        loss = criterion(
            logits,
            labels,
        )

        probs = torch.sigmoid(logits)

        preds = (
            probs >= threshold
        ).float()

        total_loss += loss.item()

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(
        all_labels,
        all_preds,
    )

    prec = precision_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    rec = recall_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    f1 = f1_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    auc = roc_auc_score(
        all_labels,
        all_probs,
    )

    return (

        avg_loss,

        acc,

        prec,

        rec,

        f1,

        auc,
    )

# ============================================================
# TRAINING
# ============================================================

def train_model(

    model,

    train_loader,

    val_loader,

    test_loader,

    epochs=60,
):

    model.to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(

        pos_weight=torch.tensor([2.0]).to(DEVICE)
    )

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=1e-4,

        weight_decay=5e-3,
    )

    scheduler = torch.optim.lr_scheduler.OneCycleLR(

        optimizer,

        max_lr=3e-4,

        epochs=epochs,

        steps_per_epoch=len(train_loader),
    )

    best_acc = 0.0

    best_threshold = 0.5

    patience = 12

    patience_counter = 0

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0

        pbar = tqdm(

            train_loader,

            desc=f"Epoch {epoch+1}/{epochs}"
        )

        for data, labels in pbar:

            data = {

                k: v.to(DEVICE)

                for k, v in data.items()
            }

            labels = labels.float().to(
                DEVICE
            ).view(-1)

            # ================================================
            # LABEL SMOOTHING
            # ================================================

            labels = labels * 0.9 + 0.05

            optimizer.zero_grad()

            logits, _, _ = model(data)

            logits = logits.view(-1)

            loss = criterion(
                logits,
                labels,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                1.0,
            )

            optimizer.step()

            scheduler.step()

            running_loss += loss.item()

            pbar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        # ====================================================
        # FIND BEST THRESHOLD
        # ====================================================

        threshold = find_best_threshold(

            model,

            val_loader,
        )

        (
            val_loss,
            val_acc,
            val_prec,
            val_rec,
            val_f1,
            val_auc,
        ) = evaluate(

            model,

            val_loader,

            criterion,

            threshold=threshold,
        )

        print("\n" + "="*70)

        print(
            f"Epoch {epoch+1}/{epochs}"
        )

        print("="*70)

        print(
            f"VAL LOSS   : {val_loss:.4f}"
        )

        print(
            f"VAL ACC    : {val_acc:.4f}"
        )

        print(
            f"VAL PREC   : {val_prec:.4f}"
        )

        print(
            f"VAL REC    : {val_rec:.4f}"
        )

        print(
            f"VAL F1     : {val_f1:.4f}"
        )

        print(
            f"VAL AUC    : {val_auc:.4f}"
        )

        # ====================================================
        # SAVE BEST MODEL
        # ====================================================

        if val_acc > best_acc:

            best_acc = val_acc

            best_threshold = threshold

            patience_counter = 0

            torch.save(

                model.state_dict(),

                "best_student_model.pth"
            )

            print(
                "\nBEST MODEL SAVED"
            )

        else:

            patience_counter += 1

            print(
                f"\nNo improvement "
                f"({patience_counter}/{patience})"
            )

            if patience_counter >= patience:

                print(
                    "\nEARLY STOPPING"
                )

                break

    # ========================================================
    # LOAD BEST MODEL
    # ========================================================

    model.load_state_dict(

        torch.load(
            "best_student_model.pth"
        )
    )

    # ========================================================
    # FINAL TEST
    # ========================================================

    (
        test_loss,
        test_acc,
        test_prec,
        test_rec,
        test_f1,
        test_auc,
    ) = evaluate(

        model,

        test_loader,

        criterion,

        threshold=best_threshold,
    )

    print("\n" + "="*70)

    print("FINAL TEST RESULTS")

    print("="*70)

    print(f"Accuracy  : {test_acc:.4f}")

    print(f"Precision : {test_prec:.4f}")

    print(f"Recall    : {test_rec:.4f}")

    print(f"F1 Score  : {test_f1:.4f}")

    print(f"AUC       : {test_auc:.4f}")

    print(f"Threshold : {best_threshold:.2f}")

    print("="*70)

    return model

# ============================================================
# TRAIN STUDENT
# ============================================================

student_model = FinalITStudent(

    embed_dim=128,

    depth=5,

    heads=8,

    dropout=0.1,
)

trained_student = train_model(

    model=student_model,

    train_loader=train_loader,

    val_loader=val_loader,

    test_loader=test_loader,

    epochs=60,
)


Using Device: cuda


Epoch 1/60: 100%|██████████| 214/214 [00:05<00:00, 36.54it/s, loss=0.7298]



Best Threshold: 0.64
Best Validation Accuracy: 0.7558

Epoch 1/60
VAL LOSS   : 0.7909
VAL ACC    : 0.7558
VAL PREC   : 0.6712
VAL REC    : 0.1922
VAL F1     : 0.2988
VAL AUC    : 0.6929

BEST MODEL SAVED


Epoch 2/60: 100%|██████████| 214/214 [00:04<00:00, 48.22it/s, loss=0.8299]



Best Threshold: 0.70
Best Validation Accuracy: 0.7696

Epoch 2/60
VAL LOSS   : 0.7219
VAL ACC    : 0.7696
VAL PREC   : 0.6610
VAL REC    : 0.3059
VAL F1     : 0.4182
VAL AUC    : 0.7688

BEST MODEL SAVED


Epoch 3/60: 100%|██████████| 214/214 [00:04<00:00, 50.43it/s, loss=0.6116]


... (45 outputs left out)

Epoch 26/60: 100%|██████████| 214/214 [00:04<00:00, 49.40it/s, loss=0.5692]



Best Threshold: 0.38
Best Validation Accuracy: 0.8004

Epoch 26/60
VAL LOSS   : 0.7851
VAL ACC    : 0.8004
VAL PREC   : 0.6959
VAL REC    : 0.4667
VAL F1     : 0.5587
VAL AUC    : 0.8287

No improvement (11/12)


Epoch 27/60: 100%|██████████| 214/214 [00:04<00:00, 49.46it/s, loss=0.4031]



Best Threshold: 0.44
Best Validation Accuracy: 0.8089

Epoch 27/60
VAL LOSS   : 0.7630
VAL ACC    : 0.8089
VAL PREC   : 0.7246
VAL REC    : 0.4745
VAL F1     : 0.5735
VAL AUC    : 0.8237

No improvement (12/12)

EARLY STOPPING

FINAL TEST RESULTS
Accuracy  : 0.8674
Precision : 0.7053
Recall    : 0.8576
F1 Score  : 0.7740
AUC       : 0.9472
Threshold : 0.38


# Teacher Model

In [7]:
# ============================================================
# FINAL MULTIMODAL KD TEACHER
# SHARED STUDENT IMU BLOCK
# STCONVTRANSFORMER SKELETON BRANCH
# SMARTFALLMM
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"\nUsing Device: {DEVICE}")

# ============================================================
# DROPPATH
# ============================================================

class DropPath(nn.Module):

    def __init__(self, drop_prob=0.0):

        super().__init__()

        self.drop_prob = drop_prob

    def forward(self, x):

        if (
            self.drop_prob == 0.0
            or
            not self.training
        ):
            return x

        keep_prob = 1 - self.drop_prob

        shape = (
            x.shape[0],
        ) + (1,) * (x.ndim - 1)

        random_tensor = torch.empty(

            shape,

            dtype=x.dtype,

            device=x.device,

        ).bernoulli_(keep_prob)

        return x * random_tensor.div(keep_prob)

# ============================================================
# TEMPORAL ATTENTION POOLING
# ============================================================

class TemporalAttentionPooling(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.attn = nn.Sequential(

            nn.LayerNorm(dim),

            nn.Linear(dim, dim // 2),

            nn.GELU(),

            nn.Dropout(0.1),

            nn.Linear(dim // 2, 1),
        )

    def forward(self, x):

        weights = self.attn(x)

        weights = torch.softmax(
            weights,
            dim=1,
        )

        pooled = (x * weights).sum(dim=1)

        return pooled

# ============================================================
# TRANSFORMER BLOCK
# ============================================================

class ITBlock(nn.Module):

    def __init__(

        self,

        dim,

        heads,

        mlp_ratio=4,

        dropout=0.1,

        drop_path=0.1,
    ):

        super().__init__()

        self.norm1 = nn.LayerNorm(dim)

        self.attn = nn.MultiheadAttention(

            dim,

            heads,

            dropout=dropout,

            batch_first=True,
        )

        self.drop_path = DropPath(drop_path)

        self.norm2 = nn.LayerNorm(dim)

        hidden_dim = int(dim * mlp_ratio)

        self.mlp = nn.Sequential(

            nn.Linear(dim, hidden_dim),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(hidden_dim, dim),

            nn.Dropout(dropout),
        )

    def forward(self, x):

        residual = x

        x = self.norm1(x)

        x, _ = self.attn(

            x,
            x,
            x,

            need_weights=False
        )

        x = residual + self.drop_path(x)

        residual = x

        x = self.norm2(x)

        x = self.mlp(x)

        x = residual + self.drop_path(x)

        return x

# ============================================================
# SE BLOCK
# ============================================================

class SE(nn.Module):

    def __init__(

        self,

        channels,

        reduction=4,
    ):

        super().__init__()

        hidden = max(
            1,
            channels // reduction
        )

        self.fc = nn.Sequential(

            nn.Linear(channels, hidden),

            nn.GELU(),

            nn.Linear(hidden, channels),

            nn.Sigmoid(),
        )

    def forward(self, x):

        w = x.mean(dim=(-2, -1))

        w = self.fc(w)

        return x * w.unsqueeze(-1).unsqueeze(-1)

# ============================================================
# MULTISCALE SPATIAL CONV
# ============================================================

class MultiScaleSpatialConv(nn.Module):

    def __init__(

        self,

        in_ch,

        out_ch,
    ):

        super().__init__()

        mid = out_ch // 3

        self.b1 = nn.Sequential(

            nn.Conv2d(in_ch, mid, 1),

            nn.BatchNorm2d(mid),

            nn.GELU(),
        )

        self.b2 = nn.Sequential(

            nn.Conv2d(
                in_ch,
                mid,
                (1, 3),
                padding=(0, 1),
            ),

            nn.BatchNorm2d(mid),

            nn.GELU(),
        )

        self.b3 = nn.Sequential(

            nn.Conv2d(
                in_ch,
                out_ch - 2 * mid,
                (1, 5),
                padding=(0, 2),
            ),

            nn.BatchNorm2d(
                out_ch - 2 * mid
            ),

            nn.GELU(),
        )

        self.fuse = nn.Sequential(

            nn.Conv2d(
                out_ch,
                out_ch,
                1,
            ),

            nn.BatchNorm2d(out_ch),

            nn.GELU(),
        )

        self.se = SE(out_ch)

        self.residual = (

            nn.Sequential(

                nn.Conv2d(
                    in_ch,
                    out_ch,
                    1,
                ),

                nn.BatchNorm2d(out_ch),
            )

            if in_ch != out_ch

            else nn.Identity()
        )

    def forward(self, x):

        residual = self.residual(x)

        x = torch.cat(

            [
                self.b1(x),
                self.b2(x),
                self.b3(x),
            ],

            dim=1,
        )

        x = self.fuse(x)

        x = self.se(x)

        return F.gelu(x + residual)

# ============================================================
# CROSS MODAL FUSION
# ============================================================

class CrossModalFusion(nn.Module):

    def __init__(

        self,

        dim,

        heads,

        dropout=0.1,
    ):

        super().__init__()

        self.norm_s = nn.LayerNorm(dim)

        self.norm_i = nn.LayerNorm(dim)

        self.attn_si = nn.MultiheadAttention(

            dim,

            heads,

            dropout=dropout,

            batch_first=True,
        )

        self.attn_is = nn.MultiheadAttention(

            dim,

            heads,

            dropout=dropout,

            batch_first=True,
        )

        self.out_proj = nn.Sequential(

            nn.Linear(dim, dim),

            nn.LayerNorm(dim),
        )

    def forward(self, skel, imu):

        s = self.norm_s(skel)

        i = self.norm_i(imu)

        s_attn, _ = self.attn_si(
            s, i, i
        )

        i_attn, _ = self.attn_is(
            i, s, s
        )

        s_out = skel + s_attn

        i_out = imu + i_attn

        if i_out.size(1) != s_out.size(1):

            i_out = F.interpolate(

                i_out.transpose(1, 2),

                size=s_out.size(1),

                mode="linear",

                align_corners=False,
            ).transpose(1, 2)

        fused = s_out + i_out

        return self.out_proj(fused)

# ============================================================
# FINAL KD TEACHER
# ============================================================

class FinalKDTeacher(nn.Module):

    def __init__(

        self,

        frames=64,

        joints=32,

        acc_channels=4,

        embed_dim=128,

        heads=8,

        skel_depth=3,

        imu_depth=5,

        temp_depth=2,

        dropout=0.1,

        drop_path=0.15,
    ):

        super().__init__()

        self.frames = frames

        self.embed_dim = embed_dim

        self.imu_tokens = frames // 4

        # ====================================================
        # SKELETON SPATIAL BRANCH
        # ====================================================

        self.spatial = nn.Sequential(

            MultiScaleSpatialConv(
                3,
                embed_dim // 2,
            ),

            MultiScaleSpatialConv(
                embed_dim // 2,
                embed_dim,
            ),
        )

        self.skel_embed = nn.Sequential(

            nn.Linear(
                joints * embed_dim,
                embed_dim,
            ),

            nn.LayerNorm(embed_dim),
        )

        # ====================================================
        # SHARED STUDENT IMU BLOCK
        # ====================================================

        self.imu_conv_stem = nn.Sequential(

            nn.Conv1d(
                acc_channels,
                32,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm1d(32),

            nn.GELU(),

            nn.Conv1d(
                32,
                64,
                kernel_size=5,
                padding=2,
            ),

            nn.BatchNorm1d(64),

            nn.GELU(),

            nn.Conv1d(
                64,
                128,
                kernel_size=3,
                padding=1,
            ),

            nn.BatchNorm1d(128),

            nn.GELU(),
        )

        self.imu_patch_embed = nn.Sequential(

            nn.Linear(
                4 * 128,
                embed_dim,
            ),

            nn.LayerNorm(embed_dim),
        )

        # ====================================================
        # POSITIONAL EMBEDDINGS
        # ====================================================

        self.pos_skel = nn.Parameter(

            torch.zeros(
                1,
                frames,
                embed_dim,
            )
        )

        self.pos_imu = nn.Parameter(

            torch.zeros(
                1,
                self.imu_tokens,
                embed_dim,
            )
        )

        self.pos_temp = nn.Parameter(

            torch.zeros(
                1,
                frames,
                embed_dim,
            )
        )

        self.pos_drop = nn.Dropout(dropout)

        # ====================================================
        # SKELETON BLOCKS
        # ====================================================

        self.skel_blocks = nn.ModuleList([

            ITBlock(

                embed_dim,

                heads,

                dropout=dropout,

                drop_path=drop_path * 0.5,
            )

            for _ in range(skel_depth)
        ])

        self.skel_norm = nn.LayerNorm(embed_dim)

        # ====================================================
        # IMU BLOCKS
        # ====================================================

        self.imu_blocks = nn.ModuleList([

            ITBlock(

                embed_dim,

                heads,

                dropout=dropout,

                drop_path=drop_path * 0.5,
            )

            for _ in range(imu_depth)
        ])

        self.imu_norm = nn.LayerNorm(embed_dim)

        # ====================================================
        # CROSS MODAL FUSION
        # ====================================================

        self.fusion = CrossModalFusion(

            embed_dim,

            heads,

            dropout,
        )

        # ====================================================
        # TEMPORAL BLOCKS
        # ====================================================

        self.temp_blocks = nn.ModuleList([

            ITBlock(

                embed_dim,

                heads,

                dropout=dropout,

                drop_path=drop_path,
            )

            for _ in range(temp_depth)
        ])

        self.temp_norm = nn.LayerNorm(embed_dim)

        # ====================================================
        # POOLING
        # ====================================================

        self.pool = TemporalAttentionPooling(
            embed_dim
        )

        # ====================================================
        # KD FEATURE HEAD
        # ====================================================

        self.feature_norm = nn.LayerNorm(
            embed_dim
        )

        self.feature_dropout = nn.Dropout(
            0.2
        )

        # ====================================================
        # CLASSIFIER
        # ====================================================

        self.head = nn.Sequential(

            nn.Linear(
                embed_dim,
                embed_dim // 2,
            ),

            nn.GELU(),

            nn.Dropout(0.1),

            nn.Linear(
                embed_dim // 2,
                1,
            ),
        )

        self._init_weights()

    # ========================================================
    # INIT
    # ========================================================

    def _init_weights(self):

        for m in self.modules():

            if isinstance(m, nn.Linear):

                nn.init.trunc_normal_(
                    m.weight,
                    std=0.02,
                )

                if m.bias is not None:

                    nn.init.constant_(
                        m.bias,
                        0,
                    )

        nn.init.trunc_normal_(
            self.pos_skel,
            std=0.02,
        )

        nn.init.trunc_normal_(
            self.pos_imu,
            std=0.02,
        )

        nn.init.trunc_normal_(
            self.pos_temp,
            std=0.02,
        )

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(self, data):

        acc = data["accelerometer"]

        skl = data["skeleton"]

        # ====================================================
        # SKELETON FORMAT
        # ====================================================

        if (
            skl.ndim == 4
            and
            skl.shape[-1] == 3
        ):

            skl = skl.permute(
                0,
                3,
                1,
                2,
            )

        B, _, T, J = skl.shape

        # ====================================================
        # SKELETON BRANCH
        # ====================================================

        sf = self.spatial(skl)

        sf = sf.permute(
            0,
            2,
            3,
            1,
        )

        sf = sf.reshape(
            B,
            T,
            J * self.embed_dim,
        )

        sf = self.skel_embed(sf)

        sf = sf + self.pos_skel[:, :T]

        sf = self.pos_drop(sf)

        for blk in self.skel_blocks:

            sf = blk(sf)

        sf = self.skel_norm(sf)

        # ====================================================
        # IMU BRANCH
        # ====================================================

        imf = acc.permute(0, 2, 1)

        imf = self.imu_conv_stem(imf)

        imf = imf.permute(0, 2, 1)

        B, T2, C = imf.shape

        imf = imf.reshape(

            B,

            T2 // 4,

            4 * C,
        )

        imf = self.imu_patch_embed(imf)

        imf = imf + self.pos_imu

        imf = self.pos_drop(imf)

        for blk in self.imu_blocks:

            imf = blk(imf)

        imf = self.imu_norm(imf)

        # ====================================================
        # CROSS MODAL FUSION
        # ====================================================

        fused = self.fusion(
            sf,
            imf,
        )

        # ====================================================
        # TEMPORAL REFINEMENT
        # ====================================================

        x = fused + self.pos_temp[:, :T]

        x = self.pos_drop(x)

        for blk in self.temp_blocks:

            x = blk(x)

        x = self.temp_norm(x)

        # ====================================================
        # POOLING
        # ====================================================

        features = self.pool(x)

        # ====================================================
        # KD FEATURES
        # ====================================================

        features = self.feature_norm(
            features
        )

        features = self.feature_dropout(
            features
        )

        features = F.normalize(

            features,

            dim=-1,
        )

        # ====================================================
        # CLASSIFIER
        # ====================================================

        logits = self.head(features)

        return logits, features

# ============================================================
# FIND BEST THRESHOLD
# ============================================================

@torch.no_grad()
def find_best_threshold(

    model,

    loader,
):

    model.eval()

    all_probs = []

    all_labels = []

    for data, labels in loader:

        data = {

            k: v.to(DEVICE)

            for k, v in data.items()
        }

        logits, _ = model(data)

        probs = torch.sigmoid(
            logits.view(-1)
        )

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

    best_acc = 0.0

    best_threshold = 0.5

    thresholds = np.arange(
        0.30,
        0.81,
        0.02
    )

    for t in thresholds:

        preds = (
            np.array(all_probs) >= t
        ).astype(int)

        acc = accuracy_score(
            all_labels,
            preds,
        )

        if acc > best_acc:

            best_acc = acc

            best_threshold = t

    print(
        f"\nBest Threshold: {best_threshold:.2f}"
    )

    print(
        f"Best Validation Accuracy: {best_acc:.4f}"
    )

    return best_threshold

# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(

    model,

    loader,

    criterion,

    threshold=0.5,
):

    model.eval()

    total_loss = 0.0

    all_probs = []

    all_preds = []

    all_labels = []

    for data, labels in loader:

        data = {

            k: v.to(DEVICE)

            for k, v in data.items()
        }

        labels = labels.float().to(
            DEVICE
        ).view(-1)

        logits, _ = model(data)

        logits = logits.view(-1)

        loss = criterion(
            logits,
            labels,
        )

        probs = torch.sigmoid(logits)

        preds = (
            probs >= threshold
        ).float()

        total_loss += loss.item()

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(
        all_labels,
        all_preds,
    )

    prec = precision_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    rec = recall_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    f1 = f1_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    auc = roc_auc_score(
        all_labels,
        all_probs,
    )

    return (

        avg_loss,

        acc,

        prec,

        rec,

        f1,

        auc,
    )

# ============================================================
# TRAINING
# ============================================================

def train_teacher(

    model,

    train_loader,

    val_loader,

    test_loader,

    epochs=50,
):

    model.to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(

        pos_weight=torch.tensor([2.0]).to(DEVICE)
    )

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=2e-4,

        weight_decay=5e-3,
    )

    scheduler = torch.optim.lr_scheduler.OneCycleLR(

        optimizer,

        max_lr=5e-4,

        epochs=epochs,

        steps_per_epoch=len(train_loader),
    )

    best_acc = 0.0

    best_threshold = 0.5

    patience = 12

    patience_counter = 0

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0

        pbar = tqdm(

            train_loader,

            desc=f"Teacher Epoch {epoch+1}/{epochs}"
        )

        for data, labels in pbar:

            data = {

                k: v.to(DEVICE)

                for k, v in data.items()
            }

            labels = labels.float().to(
                DEVICE
            ).view(-1)

            labels = labels * 0.9 + 0.05

            optimizer.zero_grad()

            logits, _ = model(data)

            logits = logits.view(-1)

            loss = criterion(
                logits,
                labels,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
            )

            optimizer.step()

            scheduler.step()

            running_loss += loss.item()

            pbar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        threshold = find_best_threshold(

            model,

            val_loader,
        )

        (
            val_loss,
            val_acc,
            val_prec,
            val_rec,
            val_f1,
            val_auc,
        ) = evaluate(

            model,

            val_loader,

            criterion,

            threshold=threshold,
        )

        print("\n" + "="*70)

        print(
            f"Teacher Epoch {epoch+1}/{epochs}"
        )

        print("="*70)

        print(
            f"VAL LOSS   : {val_loss:.4f}"
        )

        print(
            f"VAL ACC    : {val_acc:.4f}"
        )

        print(
            f"VAL PREC   : {val_prec:.4f}"
        )

        print(
            f"VAL REC    : {val_rec:.4f}"
        )

        print(
            f"VAL F1     : {val_f1:.4f}"
        )

        print(
            f"VAL AUC    : {val_auc:.4f}"
        )

        if val_acc > best_acc:

            best_acc = val_acc

            best_threshold = threshold

            patience_counter = 0

            torch.save(

                model.state_dict(),

                "best_teacher_model.pth"
            )

            print(
                "\nBEST TEACHER SAVED"
            )

        else:

            patience_counter += 1

            print(
                f"\nNo improvement "
                f"({patience_counter}/{patience})"
            )

            if patience_counter >= patience:

                print(
                    "\nEARLY STOPPING"
                )

                break

    # ========================================================
    # LOAD BEST MODEL
    # ========================================================

    model.load_state_dict(

        torch.load(
            "best_teacher_model.pth"
        )
    )

    # ========================================================
    # FINAL TEST
    # ========================================================

    (
        test_loss,
        test_acc,
        test_prec,
        test_rec,
        test_f1,
        test_auc,
    ) = evaluate(

        model,

        test_loader,

        criterion,

        threshold=best_threshold,
    )

    print("\n" + "="*70)

    print("FINAL TEACHER RESULTS")

    print("="*70)

    print(f"Accuracy  : {test_acc:.4f}")

    print(f"Precision : {test_prec:.4f}")

    print(f"Recall    : {test_rec:.4f}")

    print(f"F1 Score  : {test_f1:.4f}")

    print(f"AUC       : {test_auc:.4f}")

    print(f"Threshold : {best_threshold:.2f}")

    print("="*70)

    return model

# ============================================================
# TRAIN TEACHER
# ============================================================

teacher_model = FinalKDTeacher(

    embed_dim=128,

    heads=8,

    skel_depth=3,

    imu_depth=5,

    temp_depth=2,

    dropout=0.1,
)

trained_teacher = train_teacher(

    model=teacher_model,

    train_loader=train_loader,

    val_loader=val_loader,

    test_loader=test_loader,

    epochs=50,
)


Using Device: cuda


Teacher Epoch 1/50: 100%|██████████| 214/214 [00:11<00:00, 18.10it/s, loss=0.8179]



Best Threshold: 0.50
Best Validation Accuracy: 0.9289

Teacher Epoch 1/50
VAL LOSS   : 0.8598
VAL ACC    : 0.9289
VAL PREC   : 0.9087
VAL REC    : 0.8196
VAL F1     : 0.8619
VAL AUC    : 0.9690

BEST TEACHER SAVED


Teacher Epoch 2/50: 100%|██████████| 214/214 [00:11<00:00, 18.56it/s, loss=0.7874]



Best Threshold: 0.48
Best Validation Accuracy: 0.9565

Teacher Epoch 2/50
VAL LOSS   : 0.7836
VAL ACC    : 0.9565
VAL PREC   : 0.9864
VAL REC    : 0.8510
VAL F1     : 0.9137
VAL AUC    : 0.9749

BEST TEACHER SAVED


Teacher Epoch 3/50: 100%|██████████| 214/214 [00:11<00:00, 18.49it/s, loss=0.6375]


... (51 outputs left out)

Teacher Epoch 29/50: 100%|██████████| 214/214 [00:11<00:00, 18.17it/s, loss=0.3101]



Best Threshold: 0.30
Best Validation Accuracy: 0.9692

Teacher Epoch 29/50
VAL LOSS   : 0.2049
VAL ACC    : 0.9692
VAL PREC   : 0.9748
VAL REC    : 0.9098
VAL F1     : 0.9412
VAL AUC    : 0.9625

No improvement (11/12)


Teacher Epoch 30/50: 100%|██████████| 214/214 [00:11<00:00, 18.20it/s, loss=0.3529]



Best Threshold: 0.34
Best Validation Accuracy: 0.9671

Teacher Epoch 30/50
VAL LOSS   : 0.2140
VAL ACC    : 0.9671
VAL PREC   : 0.9912
VAL REC    : 0.8863
VAL F1     : 0.9358
VAL AUC    : 0.9827

No improvement (12/12)

EARLY STOPPING

FINAL TEACHER RESULTS
Accuracy  : 0.9596
Precision : 0.8925
Recall    : 0.9633
F1 Score  : 0.9266
AUC       : 0.9931
Threshold : 0.30


# Knowledge Distillation

In [8]:
# ============================================================
# FINAL STABLE KD TRAINING PIPELINE
# FOR:
# 1. FinalKDTeacher
# 2. FinalITStudent
# ============================================================

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"\nUsing Device: {DEVICE}")

# ============================================================
# FINAL KD CONFIG
# ============================================================

TEMPERATURE = 2.0

ALPHA_HARD = 0.85

ALPHA_FEAT = 0.05

ALPHA_LOGIT = 0.10

FEATURE_KD_START_EPOCH = 5

KD_EPOCHS = 25

KD_LR = 3e-5

KD_PATIENCE = 10

# ============================================================
# FEATURE KD LOSS
# ============================================================

def feature_kd_loss(

    student_feat,

    teacher_feat,
):

    student_feat = F.normalize(

        student_feat,

        dim=-1,
    )

    teacher_feat = F.normalize(

        teacher_feat,

        dim=-1,
    )

    loss = 1 - F.cosine_similarity(

        student_feat,

        teacher_feat,

        dim=-1,
    )

    return loss.mean()

# ============================================================
# LOGIT KD LOSS
# ============================================================

def logit_kd_loss(

    student_logits,

    teacher_logits,

    temperature=2.0,
):

    # --------------------------------------------------------
    # BINARY -> 2 CLASS
    # --------------------------------------------------------

    student_logits = torch.stack(

        [
            -student_logits,
            student_logits,
        ],

        dim=-1,
    )

    teacher_logits = torch.stack(

        [
            -teacher_logits,
            teacher_logits,
        ],

        dim=-1,
    )

    # --------------------------------------------------------
    # SOFT TARGETS
    # --------------------------------------------------------

    s = F.log_softmax(

        student_logits / temperature,

        dim=-1,
    )

    t = F.softmax(

        teacher_logits / temperature,

        dim=-1,
    )

    # --------------------------------------------------------
    # KL DIVERGENCE
    # --------------------------------------------------------

    loss = F.kl_div(

        s,

        t,

        reduction="batchmean",
    )

    return loss * (temperature ** 2)

# ============================================================
# FIND BEST THRESHOLD
# ============================================================

@torch.no_grad()
def find_best_threshold(

    model,

    loader,
):

    model.eval()

    all_probs = []

    all_labels = []

    for data, labels in loader:

        data = {

            k: v.to(DEVICE)

            for k, v in data.items()
        }

        logits, _, _ = model(data)

        probs = torch.sigmoid(
            logits.view(-1)
        )

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

    best_acc = 0.0

    best_threshold = 0.5

    thresholds = np.arange(
        0.30,
        0.81,
        0.02,
    )

    for t in thresholds:

        preds = (
            np.array(all_probs) >= t
        ).astype(int)

        acc = accuracy_score(

            all_labels,

            preds,
        )

        if acc > best_acc:

            best_acc = acc

            best_threshold = t

    print(
        f"\nBest Threshold: {best_threshold:.2f}"
    )

    print(
        f"Best Validation Accuracy: "
        f"{best_acc:.4f}"
    )

    return best_threshold

# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_kd(

    model,

    loader,

    criterion,

    threshold=0.5,
):

    model.eval()

    total_loss = 0.0

    all_probs = []

    all_preds = []

    all_labels = []

    for data, labels in loader:

        data = {

            k: v.to(DEVICE)

            for k, v in data.items()
        }

        labels = labels.float().to(
            DEVICE
        ).view(-1)

        logits, _, _ = model(data)

        logits = logits.view(-1)

        loss = criterion(
            logits,
            labels,
        )

        probs = torch.sigmoid(logits)

        preds = (
            probs >= threshold
        ).float()

        total_loss += loss.item()

        all_probs.extend(
            probs.detach().cpu().numpy()
        )

        all_preds.extend(
            preds.detach().cpu().numpy()
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(
        all_labels,
        all_preds,
    )

    prec = precision_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    rec = recall_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    f1 = f1_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    auc = roc_auc_score(
        all_labels,
        all_probs,
    )

    return (

        avg_loss,

        acc,

        prec,

        rec,

        f1,

        auc,
    )

# ============================================================
# FINAL KD TRAINING
# ============================================================

def train_kd(

    teacher,

    student,

    train_loader,

    val_loader,

    test_loader,

    epochs=KD_EPOCHS,
):

    # ========================================================
    # DEVICE
    # ========================================================

    teacher.to(DEVICE)

    student.to(DEVICE)

    # ========================================================
    # FREEZE TEACHER
    # ========================================================

    teacher.eval()

    for p in teacher.parameters():

        p.requires_grad = False

    # ========================================================
    # HARD LOSS
    # ========================================================

    hard_criterion = nn.BCEWithLogitsLoss(

        pos_weight=torch.tensor([2.0]).to(DEVICE)
    )

    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = torch.optim.AdamW(

        student.parameters(),

        lr=KD_LR,

        weight_decay=5e-3,
    )

    # ========================================================
    # SCHEDULER
    # ========================================================

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(

        optimizer,

        T_0=5,

        T_mult=2,
    )

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    best_acc = 0.0

    best_threshold = 0.5

    patience = KD_PATIENCE

    patience_counter = 0

    # ========================================================
    # TRAIN LOOP
    # ========================================================

    for epoch in range(epochs):

        student.train()

        running_total = 0.0

        running_hard = 0.0

        running_feat = 0.0

        running_logit = 0.0

        pbar = tqdm(

            train_loader,

            desc=f"KD Epoch {epoch+1}/{epochs}"
        )

        for data, labels in pbar:

            # ------------------------------------------------
            # DEVICE
            # ------------------------------------------------

            data = {

                k: v.to(DEVICE)

                for k, v in data.items()
            }

            labels = labels.float().to(
                DEVICE
            ).view(-1)

            # ------------------------------------------------
            # LABEL SMOOTHING
            # ------------------------------------------------

            labels = labels * 0.9 + 0.05

            # =================================================
            # TEACHER FORWARD
            # =================================================

            with torch.no_grad():

                teacher_logits, teacher_feat = \
                    teacher(data)

                teacher_logits = \
                    teacher_logits.view(-1)

            # =================================================
            # STUDENT FORWARD
            # =================================================

            student_logits, student_feat, _ = \
                student(data)

            student_logits = \
                student_logits.view(-1)

            # =================================================
            # HARD LOSS
            # =================================================

            hard_loss = hard_criterion(

                student_logits,

                labels,
            )

            # =================================================
            # FEATURE KD
            # =================================================

            if epoch >= FEATURE_KD_START_EPOCH:

                feat_loss = feature_kd_loss(

                    student_feat,

                    teacher_feat,
                )

            else:

                feat_loss = torch.tensor(

                    0.0,

                    device=DEVICE,
                )

            # =================================================
            # LOGIT KD
            # =================================================

            kd_loss = logit_kd_loss(

                student_logits,

                teacher_logits,

                TEMPERATURE,
            )

            # =================================================
            # TOTAL LOSS
            # =================================================

            total_loss = (

                ALPHA_HARD * hard_loss

                +

                ALPHA_FEAT * feat_loss

                +

                ALPHA_LOGIT * kd_loss
            )

            # =================================================
            # OPTIMIZATION
            # =================================================

            optimizer.zero_grad()

            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(

                student.parameters(),

                1.0,
            )

            optimizer.step()

            # =================================================
            # TRACKING
            # =================================================

            running_total += total_loss.item()

            running_hard += hard_loss.item()

            running_feat += feat_loss.item()

            running_logit += kd_loss.item()

            pbar.set_postfix(

                total=f"{total_loss.item():.4f}",

                hard=f"{hard_loss.item():.4f}",

                feat=f"{feat_loss.item():.4f}",

                kd=f"{kd_loss.item():.4f}",
            )

        # ====================================================
        # LR SCHEDULER
        # ====================================================

        scheduler.step()

        # ====================================================
        # FIND BEST THRESHOLD
        # ====================================================

        threshold = find_best_threshold(

            student,

            val_loader,
        )

        # ====================================================
        # VALIDATION
        # ====================================================

        (
            val_loss,
            val_acc,
            val_prec,
            val_rec,
            val_f1,
            val_auc,
        ) = evaluate_kd(

            student,

            val_loader,

            hard_criterion,

            threshold=threshold,
        )

        # ====================================================
        # AVERAGE LOSSES
        # ====================================================

        avg_total = (
            running_total / len(train_loader)
        )

        avg_hard = (
            running_hard / len(train_loader)
        )

        avg_feat = (
            running_feat / len(train_loader)
        )

        avg_logit = (
            running_logit / len(train_loader)
        )

        # ====================================================
        # PRINT
        # ====================================================

        print("\n" + "="*70)

        print(
            f"KD Epoch {epoch+1}/{epochs}"
        )

        print("="*70)

        print(
            f"TOTAL LOSS : {avg_total:.4f}"
        )

        print(
            f"HARD LOSS  : {avg_hard:.4f}"
        )

        print(
            f"FEAT LOSS  : {avg_feat:.4f}"
        )

        print(
            f"LOGIT LOSS : {avg_logit:.4f}"
        )

        print("-"*70)

        print(
            f"VAL LOSS : {val_loss:.4f}"
        )

        print(
            f"VAL ACC  : {val_acc:.4f}"
        )

        print(
            f"VAL PREC : {val_prec:.4f}"
        )

        print(
            f"VAL REC  : {val_rec:.4f}"
        )

        print(
            f"VAL F1   : {val_f1:.4f}"
        )

        print(
            f"VAL AUC  : {val_auc:.4f}"
        )

        print(
            f"THRESHOLD: {threshold:.2f}"
        )

        # ====================================================
        # SAVE BEST MODEL
        # ====================================================

        if val_acc > best_acc:

            best_acc = val_acc

            best_threshold = threshold

            patience_counter = 0

            torch.save(

                student.state_dict(),

                "best_kd_student.pth"
            )

            print(
                "\nBEST KD STUDENT SAVED"
            )

        else:

            patience_counter += 1

            print(
                f"\nNo improvement "
                f"({patience_counter}/"
                f"{patience})"
            )

            if patience_counter >= patience:

                print(
                    "\nEARLY STOPPING"
                )

                break

    # ========================================================
    # LOAD BEST STUDENT
    # ========================================================

    print("\nLoading best KD student...")

    student.load_state_dict(

        torch.load(
            "best_kd_student.pth"
        )
    )

    # ========================================================
    # FINAL TEST
    # ========================================================

    (
        test_loss,
        test_acc,
        test_prec,
        test_rec,
        test_f1,
        test_auc,
    ) = evaluate_kd(

        student,

        test_loader,

        hard_criterion,

        threshold=best_threshold,
    )

    print("\n" + "="*70)

    print("FINAL KD RESULTS")

    print("="*70)

    print(f"Accuracy  : {test_acc:.4f}")

    print(f"Precision : {test_prec:.4f}")

    print(f"Recall    : {test_rec:.4f}")

    print(f"F1 Score  : {test_f1:.4f}")

    print(f"AUC       : {test_auc:.4f}")

    print(f"Threshold : {best_threshold:.2f}")

    print("="*70)

    return student

# ============================================================
# LOAD TEACHER
# ============================================================

teacher_model = FinalKDTeacher(

    embed_dim=128,

    heads=8,

    skel_depth=3,

    imu_depth=5,

    temp_depth=2,

    dropout=0.1,
)

teacher_model.load_state_dict(

    torch.load(
        "best_teacher_model.pth"
    )
)

print("\nTeacher loaded successfully.")

# ============================================================
# LOAD PRETRAINED STUDENT
# ============================================================

student_model = FinalITStudent(

    embed_dim=128,

    depth=5,

    heads=8,

    dropout=0.1,
)

student_model.load_state_dict(

    torch.load(
        "best_student_model.pth"
    )
)

print(
    "\nPretrained student loaded successfully."
)

# ============================================================
# TRAIN KD
# ============================================================

kd_student = train_kd(

    teacher=teacher_model,

    student=student_model,

    train_loader=train_loader,

    val_loader=val_loader,

    test_loader=test_loader,

    epochs=KD_EPOCHS,
)


Using Device: cuda

Teacher loaded successfully.

Pretrained student loaded successfully.


KD Epoch 1/25: 100%|██████████| 214/214 [00:06<00:00, 33.60it/s, feat=0.0000, hard=0.3319, kd=0.0830, total=0.2904]



Best Threshold: 0.42
Best Validation Accuracy: 0.8206

KD Epoch 1/25
TOTAL LOSS : 0.4214
HARD LOSS  : 0.4478
FEAT LOSS  : 0.0000
LOGIT LOSS : 0.4077
----------------------------------------------------------------------
VAL LOSS : 0.7231
VAL ACC  : 0.8206
VAL PREC : 0.7363
VAL REC  : 0.5255
VAL F1   : 0.6133
VAL AUC  : 0.8471
THRESHOLD: 0.42

BEST KD STUDENT SAVED


KD Epoch 2/25: 100%|██████████| 214/214 [00:06<00:00, 34.26it/s, feat=0.0000, hard=0.3256, kd=0.0221, total=0.2790]



Best Threshold: 0.44
Best Validation Accuracy: 0.8248

KD Epoch 2/25
TOTAL LOSS : 0.3990
HARD LOSS  : 0.4265
FEAT LOSS  : 0.0000
LOGIT LOSS : 0.3642
----------------------------------------------------------------------
VAL LOSS : 0.7112
VAL ACC  : 0.8248
VAL PREC : 0.7557
VAL REC  : 0.5216
VAL F1   : 0.6172
VAL AUC  : 0.8396
THRESHOLD: 0.44

BEST KD STUDENT SAVED


KD Epoch 3/25: 100%|██████████| 214/214 [00:06<00:00, 34.14it/s, feat=0.0000, hard=0.3318, kd=0.1030, total=0.2923]


... (15 outputs left out)

KD Epoch 11/25: 100%|██████████| 214/214 [00:06<00:00, 32.87it/s, feat=0.0873, hard=0.3222, kd=0.0487, total=0.2831]



Best Threshold: 0.42
Best Validation Accuracy: 0.8089

KD Epoch 11/25
TOTAL LOSS : 0.3682
HARD LOSS  : 0.3935
FEAT LOSS  : 0.1373
LOGIT LOSS : 0.2684
----------------------------------------------------------------------
VAL LOSS : 0.8221
VAL ACC  : 0.8089
VAL PREC : 0.7389
VAL REC  : 0.4549
VAL F1   : 0.5631
VAL AUC  : 0.7892
THRESHOLD: 0.42

No improvement (9/10)


KD Epoch 12/25: 100%|██████████| 214/214 [00:06<00:00, 32.62it/s, feat=0.0905, hard=0.3135, kd=0.0800, total=0.2790]



Best Threshold: 0.34
Best Validation Accuracy: 0.8121

KD Epoch 12/25
TOTAL LOSS : 0.3638
HARD LOSS  : 0.3897
FEAT LOSS  : 0.1298
LOGIT LOSS : 0.2609
----------------------------------------------------------------------
VAL LOSS : 0.7972
VAL ACC  : 0.8121
VAL PREC : 0.7267
VAL REC  : 0.4902
VAL F1   : 0.5855
VAL AUC  : 0.7982
THRESHOLD: 0.34

No improvement (10/10)

EARLY STOPPING

Loading best KD student...

FINAL KD RESULTS
Accuracy  : 0.8900
Precision : 0.7564
Recall    : 0.8620
F1 Score  : 0.8058
AUC       : 0.9559
Threshold : 0.44


In [9]:
# ============================================================
# LIGHTHART v4.1
# FINAL STABLE VERSION
# FIXED FOCAL LOSS COLLAPSE
# ============================================================

import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# ============================================================
# SEED
# ============================================================

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True

torch.backends.cudnn.benchmark = False

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"\nUsing Device: {DEVICE}")

# ============================================================
# DROPPATH
# ============================================================

class DropPath(nn.Module):

    def __init__(self, drop_prob=0.0):

        super().__init__()

        self.drop_prob = drop_prob

    def forward(self, x):

        if (
            self.drop_prob == 0.0
            or
            not self.training
        ):
            return x

        keep_prob = 1 - self.drop_prob

        shape = (
            x.shape[0],
        ) + (1,) * (x.ndim - 1)

        random_tensor = torch.empty(

            shape,

            dtype=x.dtype,

            device=x.device,

        ).bernoulli_(keep_prob)

        return x * random_tensor.div(keep_prob)

# ============================================================
# STABLE FOCAL LOSS
# ============================================================

class FocalLoss(nn.Module):

    def __init__(

        self,

        alpha=0.80,

        gamma=1.5,
    ):

        super().__init__()

        self.alpha = alpha

        self.gamma = gamma

    def forward(self, logits, targets):

        bce = F.binary_cross_entropy_with_logits(

            logits,

            targets,

            reduction="none",
        )

        probs = torch.sigmoid(logits)

        pt = torch.where(

            targets == 1,

            probs,

            1 - probs,
        )

        focal_weight = (

            1 - pt

        ) ** self.gamma

        loss = self.alpha * focal_weight * bce

        return loss.mean()

# ============================================================
# RESIDUAL DEPTHWISE BLOCK
# ============================================================

class ResidualDWBlock(nn.Module):

    def __init__(self, channels):

        super().__init__()

        self.dw = nn.Conv1d(

            channels,

            channels,

            kernel_size=3,

            padding=1,

            groups=channels,

            bias=False,
        )

        self.pw = nn.Conv1d(

            channels,

            channels,

            kernel_size=1,

            bias=False,
        )

        self.bn = nn.BatchNorm1d(channels)

        self.act = nn.GELU()

    def forward(self, x):

        residual = x

        x = self.dw(x)

        x = self.pw(x)

        x = self.bn(x)

        x = self.act(x)

        return x + residual

# ============================================================
# TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(

        self,

        embed_dim,

        num_heads,

        dropout=0.08,

        mlp_ratio=2,

        drop_path=0.08,
    ):

        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)

        self.attn = nn.MultiheadAttention(

            embed_dim,

            num_heads,

            dropout=dropout,

            batch_first=True,
        )

        self.drop_path = DropPath(drop_path)

        self.norm2 = nn.LayerNorm(embed_dim)

        hidden_dim = embed_dim * mlp_ratio

        self.ffn = nn.Sequential(

            nn.Linear(embed_dim, hidden_dim),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(hidden_dim, embed_dim),

            nn.Dropout(dropout),
        )

    def forward(self, x):

        residual = x

        x_norm = self.norm1(x)

        attn_out, _ = self.attn(

            x_norm,

            x_norm,

            x_norm,

            need_weights=False,
        )

        x = residual + self.drop_path(attn_out)

        residual = x

        x_norm = self.norm2(x)

        ffn_out = self.ffn(x_norm)

        x = residual + self.drop_path(ffn_out)

        return x

# ============================================================
# LIGHTHART v4.1
# ============================================================

class LightHART(nn.Module):

    def __init__(

        self,

        acc_frames=64,

        acc_coords=4,

        embed_dim=128,

        num_heads=8,

        num_layers=5,

        dropout=0.08,
    ):

        super().__init__()

        self.embed_dim = embed_dim

        # ====================================================
        # CNN STEM
        # ====================================================

        self.conv_stem = nn.Sequential(

            nn.Conv1d(

                acc_coords,

                64,

                kernel_size=5,

                padding=2,

                bias=False,
            ),

            nn.BatchNorm1d(64),

            nn.GELU(),

            ResidualDWBlock(64),

            nn.Conv1d(

                64,

                embed_dim,

                kernel_size=3,

                padding=1,

                bias=False,
            ),

            nn.BatchNorm1d(embed_dim),

            nn.GELU(),

            ResidualDWBlock(embed_dim),
        )

        # ====================================================
        # POSITIONAL EMBEDDINGS
        # ====================================================

        self.pos_embed = nn.Parameter(

            torch.zeros(

                1,

                acc_frames,

                embed_dim,
            )
        )

        self.pos_drop = nn.Dropout(dropout)

        # ====================================================
        # TRANSFORMER
        # ====================================================

        dpr = [

            x.item()

            for x in torch.linspace(
                0,
                0.08,
                num_layers,
            )
        ]

        self.encoder_blocks = nn.ModuleList([

            TransformerBlock(

                embed_dim=embed_dim,

                num_heads=num_heads,

                dropout=dropout,

                drop_path=dpr[i],
            )

            for i in range(num_layers)
        ])

        self.temporal_norm = nn.LayerNorm(
            embed_dim
        )

        # ====================================================
        # TEMPORAL REFINEMENT
        # ====================================================

        self.temporal_refine = nn.Sequential(

            nn.Conv1d(

                embed_dim,

                embed_dim,

                kernel_size=3,

                padding=1,

                groups=embed_dim,
            ),

            nn.BatchNorm1d(embed_dim),

            nn.GELU(),

            nn.Conv1d(

                embed_dim,

                embed_dim,

                kernel_size=1,
            ),
        )

        # ====================================================
        # ATTENTION POOLING
        # ====================================================

        self.pool_attn = nn.Sequential(

            nn.Linear(
                embed_dim,
                embed_dim // 2,
            ),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                embed_dim // 2,
                1,
            ),
        )

        # ====================================================
        # KD PROJECTOR
        # ====================================================

        self.kd_projector = nn.Sequential(

            nn.Linear(
                embed_dim,
                128,
            ),

            nn.LayerNorm(128),
        )

        # ====================================================
        # CLASSIFIER
        # ====================================================

        self.classifier = nn.Sequential(

            nn.Linear(

                embed_dim,

                embed_dim // 2,
            ),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                embed_dim // 2,
                1,
            ),
        )

        nn.init.trunc_normal_(
            self.pos_embed,
            std=0.02,
        )

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(self, data):

        x = data["accelerometer"]

        # ====================================================
        # AUGMENTATION
        # ====================================================

        if self.training:

            noise = torch.randn_like(x) * 0.015

            scale = 1.0 + (

                torch.randn(

                    x.size(0),

                    1,

                    1,

                    device=x.device
                ) * 0.05
            )

            x = x * scale + noise

        # ====================================================
        # CNN STEM
        # ====================================================

        x = x.permute(0, 2, 1)

        x = self.conv_stem(x)

        x = x.permute(0, 2, 1)

        # ====================================================
        # POSITIONAL EMBEDDINGS
        # ====================================================

        x = x + self.pos_embed

        x = self.pos_drop(x)

        # ====================================================
        # TRANSFORMER
        # ====================================================

        for encoder in self.encoder_blocks:

            x = encoder(x)

        x = self.temporal_norm(x)

        # ====================================================
        # TEMPORAL REFINEMENT
        # ====================================================

        temp = x.permute(0, 2, 1)

        temp = self.temporal_refine(temp)

        temp = temp.permute(0, 2, 1)

        x = x + temp

        features = x

        # ====================================================
        # ATTENTION POOLING
        # ====================================================

        weights = self.pool_attn(x)

        weights = torch.softmax(
            weights,
            dim=1,
        )

        pooled = (x * weights).sum(dim=1)

        # ====================================================
        # KD FEATURES
        # ====================================================

        kd_features = self.kd_projector(
            pooled
        )

        kd_features = F.normalize(

            kd_features,

            dim=-1,
        )

        # ====================================================
        # CLASSIFICATION
        # ====================================================

        logits = self.classifier(
            pooled
        )

        return logits, kd_features, features

# ============================================================
# DYNAMIC THRESHOLD
# ============================================================

@torch.no_grad()
def find_best_threshold(

    model,

    loader,
):

    model.eval()

    all_probs = []

    all_labels = []

    for data, labels in loader:

        data = {

            k: v.to(DEVICE)

            for k, v in data.items()
        }

        logits, _, _ = model(data)

        probs = torch.sigmoid(
            logits.view(-1)
        )

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

    best_acc = 0.0

    best_threshold = 0.5

    thresholds = np.arange(
        0.30,
        0.81,
        0.02,
    )

    for t in thresholds:

        preds = (
            np.array(all_probs) >= t
        ).astype(int)

        acc = accuracy_score(
            all_labels,
            preds,
        )

        if acc > best_acc:

            best_acc = acc

            best_threshold = t

    print(
        f"\nBest Threshold: {best_threshold:.2f}"
    )

    print(
        f"Best Validation Accuracy: {best_acc:.4f}"
    )

    return best_threshold

# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(

    model,

    loader,

    criterion,

    threshold=0.5,
):

    model.eval()

    total_loss = 0.0

    all_probs = []

    all_preds = []

    all_labels = []

    for data, labels in loader:

        data = {

            k: v.to(DEVICE)

            for k, v in data.items()
        }

        labels = labels.float().to(
            DEVICE
        ).view(-1)

        logits, _, _ = model(data)

        logits = logits.view(-1)

        loss = criterion(
            logits,
            labels,
        )

        probs = torch.sigmoid(logits)

        preds = (
            probs >= threshold
        ).float()

        total_loss += loss.item()

        all_probs.extend(
            probs.detach().cpu().numpy()
        )

        all_preds.extend(
            preds.detach().cpu().numpy()
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(
        all_labels,
        all_preds,
    )

    prec = precision_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    rec = recall_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    f1 = f1_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    auc = roc_auc_score(
        all_labels,
        all_probs,
    )

    return (
        avg_loss,
        acc,
        prec,
        rec,
        f1,
        auc,
    )

# ============================================================
# TRAINING
# ============================================================

def train_model(

    model,

    train_loader,

    val_loader,

    test_loader,

    epochs=60,
):

    model.to(DEVICE)

    criterion = FocalLoss(

        alpha=0.80,

        gamma=1.5,
    )

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=1e-4,

        weight_decay=3e-3,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(

        optimizer,

        T_0=5,

        T_mult=2,
    )

    best_acc = 0.0

    best_threshold = 0.5

    patience = 15

    patience_counter = 0

    # ========================================================
    # TRAIN LOOP
    # ========================================================

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0

        pbar = tqdm(

            train_loader,

            desc=f"Epoch {epoch+1}/{epochs}"
        )

        for data, labels in pbar:

            data = {

                k: v.to(DEVICE)

                for k, v in data.items()
            }

            labels = labels.float().to(
                DEVICE
            ).view(-1)

            optimizer.zero_grad()

            logits, _, _ = model(data)

            logits = logits.view(-1)

            loss = criterion(
                logits,
                labels,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                1.0,
            )

            optimizer.step()

            running_loss += loss.item()

            pbar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        scheduler.step()

        threshold = find_best_threshold(

            model,

            val_loader,
        )

        (
            val_loss,
            val_acc,
            val_prec,
            val_rec,
            val_f1,
            val_auc,
        ) = evaluate(

            model,

            val_loader,

            criterion,

            threshold=threshold,
        )

        print("\n" + "="*70)

        print(
            f"Epoch {epoch+1}/{epochs}"
        )

        print("="*70)

        print(
            f"VAL LOSS   : {val_loss:.4f}"
        )

        print(
            f"VAL ACC    : {val_acc:.4f}"
        )

        print(
            f"VAL PREC   : {val_prec:.4f}"
        )

        print(
            f"VAL REC    : {val_rec:.4f}"
        )

        print(
            f"VAL F1     : {val_f1:.4f}"
        )

        print(
            f"VAL AUC    : {val_auc:.4f}"
        )

        print(
            f"THRESHOLD  : {threshold:.2f}"
        )

        # ====================================================
        # SAVE BEST MODEL
        # ====================================================

        if val_acc > best_acc:

            best_acc = val_acc

            best_threshold = threshold

            patience_counter = 0

            torch.save(

                model.state_dict(),

                "best_lighthart_v41.pth"
            )

            print(
                "\nBEST MODEL SAVED"
            )

        else:

            patience_counter += 1

            print(
                f"\nNo improvement "
                f"({patience_counter}/{patience})"
            )

            if patience_counter >= patience:

                print(
                    "\nEARLY STOPPING"
                )

                break

    # ========================================================
    # LOAD BEST MODEL
    # ========================================================

    model.load_state_dict(

        torch.load(
            "best_lighthart_v41.pth"
        )
    )

    # ========================================================
    # FINAL TEST
    # ========================================================

    (
        test_loss,
        test_acc,
        test_prec,
        test_rec,
        test_f1,
        test_auc,
    ) = evaluate(

        model,

        test_loader,

        criterion,

        threshold=best_threshold,
    )

    print("\n" + "="*70)

    print("FINAL TEST RESULTS")

    print("="*70)

    print(f"Accuracy  : {test_acc:.4f}")

    print(f"Precision : {test_prec:.4f}")

    print(f"Recall    : {test_rec:.4f}")

    print(f"F1 Score  : {test_f1:.4f}")

    print(f"AUC       : {test_auc:.4f}")

    print(f"Threshold : {best_threshold:.2f}")

    print("="*70)

    return model

# ============================================================
# TRAIN MODEL
# ============================================================

lighthart_model = LightHART(

    embed_dim=128,

    num_heads=8,

    num_layers=5,

    dropout=0.08,
)

trained_model = train_model(

    model=lighthart_model,

    train_loader=train_loader,

    val_loader=val_loader,

    test_loader=test_loader,

    epochs=60,
)


Using Device: cuda


Epoch 1/60: 100%|██████████| 214/214 [00:04<00:00, 44.46it/s, loss=0.1001]



Best Threshold: 0.52
Best Validation Accuracy: 0.7760

Epoch 1/60
VAL LOSS   : 0.1425
VAL ACC    : 0.7760
VAL PREC   : 0.6930
VAL REC    : 0.3098
VAL F1     : 0.4282
VAL AUC    : 0.7937
THRESHOLD  : 0.52

BEST MODEL SAVED


Epoch 2/60: 100%|██████████| 214/214 [00:04<00:00, 46.22it/s, loss=0.0978]



Best Threshold: 0.58
Best Validation Accuracy: 0.7845

Epoch 2/60
VAL LOSS   : 0.1315
VAL ACC    : 0.7845
VAL PREC   : 0.7500
VAL REC    : 0.3059
VAL F1     : 0.4345
VAL AUC    : 0.8379
THRESHOLD  : 0.58

BEST MODEL SAVED


Epoch 3/60: 100%|██████████| 214/214 [00:04<00:00, 47.13it/s, loss=0.1275]


... (63 outputs left out)

Epoch 35/60: 100%|██████████| 214/214 [00:04<00:00, 47.80it/s, loss=0.0109]



Best Threshold: 0.46
Best Validation Accuracy: 0.8004

Epoch 35/60
VAL LOSS   : 0.3282
VAL ACC    : 0.8004
VAL PREC   : 0.7030
VAL REC    : 0.4549
VAL F1     : 0.5524
VAL AUC    : 0.8369
THRESHOLD  : 0.46

No improvement (14/15)


Epoch 36/60: 100%|██████████| 214/214 [00:04<00:00, 46.15it/s, loss=0.0178]



Best Threshold: 0.62
Best Validation Accuracy: 0.8047

Epoch 36/60
VAL LOSS   : 0.2371
VAL ACC    : 0.8047
VAL PREC   : 0.7351
VAL REC    : 0.4353
VAL F1     : 0.5468
VAL AUC    : 0.8440
THRESHOLD  : 0.62

No improvement (15/15)

EARLY STOPPING

FINAL TEST RESULTS
Accuracy  : 0.8970
Precision : 0.7751
Recall    : 0.8605
F1 Score  : 0.8156
AUC       : 0.9556
Threshold : 0.44


In [10]:
# ============================================================
# FINAL SHARED-CNN MULTIMODAL TEACHER v3
# COMPLETE END-TO-END TRAINING PIPELINE
# FULLY ALIGNED WITH LIGHTHART v4.1
# ============================================================

import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# ============================================================
# SEED
# ============================================================

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True

torch.backends.cudnn.benchmark = False

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"\nUsing Device: {DEVICE}")

# ============================================================
# DROPPATH
# ============================================================

class DropPath(nn.Module):

    def __init__(self, drop_prob=0.0):

        super().__init__()

        self.drop_prob = drop_prob

    def forward(self, x):

        if (
            self.drop_prob == 0.0
            or
            not self.training
        ):
            return x

        keep_prob = 1 - self.drop_prob

        shape = (
            x.shape[0],
        ) + (1,) * (x.ndim - 1)

        random_tensor = torch.empty(

            shape,

            dtype=x.dtype,

            device=x.device,

        ).bernoulli_(keep_prob)

        return x * random_tensor.div(keep_prob)

# ============================================================
# FOCAL LOSS
# ============================================================

class FocalLoss(nn.Module):

    def __init__(

        self,

        alpha=0.80,

        gamma=1.5,
    ):

        super().__init__()

        self.alpha = alpha

        self.gamma = gamma

    def forward(self, logits, targets):

        bce = F.binary_cross_entropy_with_logits(

            logits,

            targets,

            reduction="none",
        )

        probs = torch.sigmoid(logits)

        pt = torch.where(

            targets == 1,

            probs,

            1 - probs,
        )

        focal_weight = (

            1 - pt

        ) ** self.gamma

        loss = self.alpha * focal_weight * bce

        return loss.mean()

# ============================================================
# RESIDUAL DEPTHWISE BLOCK
# ============================================================

class ResidualDWBlock(nn.Module):

    def __init__(self, channels):

        super().__init__()

        self.dw = nn.Conv1d(

            channels,

            channels,

            kernel_size=3,

            padding=1,

            groups=channels,

            bias=False,
        )

        self.pw = nn.Conv1d(

            channels,

            channels,

            kernel_size=1,

            bias=False,
        )

        self.bn = nn.BatchNorm1d(channels)

        self.act = nn.GELU()

    def forward(self, x):

        residual = x

        x = self.dw(x)

        x = self.pw(x)

        x = self.bn(x)

        x = self.act(x)

        return x + residual

# ============================================================
# SE BLOCK
# ============================================================

class SE(nn.Module):

    def __init__(

        self,

        channels,

        reduction=4,
    ):

        super().__init__()

        hidden = max(
            1,
            channels // reduction
        )

        self.fc = nn.Sequential(

            nn.Linear(channels, hidden),

            nn.GELU(),

            nn.Linear(hidden, channels),

            nn.Sigmoid(),
        )

    def forward(self, x):

        w = x.mean(dim=(-2, -1))

        w = self.fc(w)

        return x * w.unsqueeze(-1).unsqueeze(-1)

# ============================================================
# MULTISCALE SPATIAL CONV
# ============================================================

class MultiScaleSpatialConv(nn.Module):

    def __init__(

        self,

        in_ch,

        out_ch,
    ):

        super().__init__()

        mid = out_ch // 3

        self.b1 = nn.Sequential(

            nn.Conv2d(in_ch, mid, 1),

            nn.BatchNorm2d(mid),

            nn.GELU(),
        )

        self.b2 = nn.Sequential(

            nn.Conv2d(

                in_ch,

                mid,

                (1, 3),

                padding=(0, 1),
            ),

            nn.BatchNorm2d(mid),

            nn.GELU(),
        )

        self.b3 = nn.Sequential(

            nn.Conv2d(

                in_ch,

                out_ch - 2 * mid,

                (1, 5),

                padding=(0, 2),
            ),

            nn.BatchNorm2d(
                out_ch - 2 * mid
            ),

            nn.GELU(),
        )

        self.fuse = nn.Sequential(

            nn.Conv2d(
                out_ch,
                out_ch,
                1,
            ),

            nn.BatchNorm2d(out_ch),

            nn.GELU(),
        )

        self.se = SE(out_ch)

        self.residual = (

            nn.Sequential(

                nn.Conv2d(
                    in_ch,
                    out_ch,
                    1,
                ),

                nn.BatchNorm2d(out_ch),
            )

            if in_ch != out_ch

            else nn.Identity()
        )

    def forward(self, x):

        residual = self.residual(x)

        x = torch.cat(

            [
                self.b1(x),
                self.b2(x),
                self.b3(x),
            ],

            dim=1,
        )

        x = self.fuse(x)

        x = self.se(x)

        return F.gelu(x + residual)

# ============================================================
# TRANSFORMER BLOCK
# ============================================================

class ITBlock(nn.Module):

    def __init__(

        self,

        dim,

        heads,

        mlp_ratio=2,

        dropout=0.10,

        drop_path=0.10,
    ):

        super().__init__()

        self.norm1 = nn.LayerNorm(dim)

        self.attn = nn.MultiheadAttention(

            dim,

            heads,

            dropout=dropout,

            batch_first=True,
        )

        self.drop_path = DropPath(drop_path)

        self.norm2 = nn.LayerNorm(dim)

        hidden = int(dim * mlp_ratio)

        self.mlp = nn.Sequential(

            nn.Linear(dim, hidden),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(hidden, dim),

            nn.Dropout(dropout),
        )

    def forward(self, x):

        residual = x

        x = self.norm1(x)

        x, _ = self.attn(
            x,
            x,
            x,
            need_weights=False,
        )

        x = residual + self.drop_path(x)

        residual = x

        x = self.norm2(x)

        x = self.mlp(x)

        x = residual + self.drop_path(x)

        return x

# ============================================================
# CROSS MODAL FUSION
# ============================================================

class CrossModalFusion(nn.Module):

    def __init__(

        self,

        dim,

        heads,

        dropout=0.10,
    ):

        super().__init__()

        self.norm_s = nn.LayerNorm(dim)

        self.norm_i = nn.LayerNorm(dim)

        self.attn_si = nn.MultiheadAttention(

            dim,

            heads,

            dropout=dropout,

            batch_first=True,
        )

        self.attn_is = nn.MultiheadAttention(

            dim,

            heads,

            dropout=dropout,

            batch_first=True,
        )

        self.out_proj = nn.Sequential(

            nn.Linear(dim, dim),

            nn.LayerNorm(dim),
        )

    def forward(self, skel, imu):

        s = self.norm_s(skel)

        i = self.norm_i(imu)

        s_attn, _ = self.attn_si(
            s,
            i,
            i,
        )

        i_attn, _ = self.attn_is(
            i,
            s,
            s,
        )

        s_out = skel + s_attn

        i_out = imu + i_attn

        fused = s_out + i_out

        return self.out_proj(fused)

# ============================================================
# ATTENTION POOLING
# ============================================================

class TemporalAttentionPooling(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.attn = nn.Sequential(

            nn.LayerNorm(dim),

            nn.Linear(dim, dim // 2),

            nn.GELU(),

            nn.Dropout(0.10),

            nn.Linear(dim // 2, 1),
        )

    def forward(self, x):

        weights = self.attn(x)

        weights = torch.softmax(
            weights,
            dim=1,
        )

        return (x * weights).sum(dim=1)

# ============================================================
# FINAL SHARED-CNN TEACHER
# ============================================================

class SharedCNNTeacher(nn.Module):

    def __init__(

        self,

        frames=64,

        joints=32,

        acc_channels=4,

        embed_dim=128,

        heads=8,

        skel_depth=3,

        imu_depth=5,

        temp_depth=2,

        dropout=0.10,

        drop_path=0.10,
    ):

        super().__init__()

        self.embed_dim = embed_dim

        # ====================================================
        # IMU CNN STEM
        # ====================================================

        self.imu_conv_stem = nn.Sequential(

            nn.Conv1d(

                acc_channels,

                64,

                kernel_size=5,

                padding=2,

                bias=False,
            ),

            nn.BatchNorm1d(64),

            nn.GELU(),

            ResidualDWBlock(64),

            nn.Conv1d(

                64,

                embed_dim,

                kernel_size=3,

                padding=1,

                bias=False,
            ),

            nn.BatchNorm1d(embed_dim),

            nn.GELU(),

            ResidualDWBlock(embed_dim),
        )

        # ====================================================
        # SKELETON BRANCH
        # ====================================================

        self.spatial = nn.Sequential(

            MultiScaleSpatialConv(
                3,
                embed_dim // 2,
            ),

            MultiScaleSpatialConv(
                embed_dim // 2,
                embed_dim,
            ),
        )

        self.skel_embed = nn.Sequential(

            nn.Linear(
                joints * embed_dim,
                embed_dim,
            ),

            nn.LayerNorm(embed_dim),
        )

        # ====================================================
        # POSITIONAL EMBEDDINGS
        # ====================================================

        self.pos_skel = nn.Parameter(

            torch.zeros(
                1,
                frames,
                embed_dim,
            )
        )

        self.pos_imu = nn.Parameter(

            torch.zeros(
                1,
                frames,
                embed_dim,
            )
        )

        self.pos_temp = nn.Parameter(

            torch.zeros(
                1,
                frames,
                embed_dim,
            )
        )

        self.pos_drop = nn.Dropout(dropout)

        # ====================================================
        # SKELETON TRANSFORMERS
        # ====================================================

        self.skel_blocks = nn.ModuleList([

            ITBlock(

                embed_dim,

                heads,

                dropout=dropout,

                drop_path=drop_path * 0.5,
            )

            for _ in range(skel_depth)
        ])

        self.skel_norm = nn.LayerNorm(embed_dim)

        # ====================================================
        # IMU TRANSFORMERS
        # ====================================================

        self.imu_blocks = nn.ModuleList([

            ITBlock(

                embed_dim,

                heads,

                dropout=dropout,

                drop_path=drop_path * 0.5,
            )

            for _ in range(imu_depth)
        ])

        self.imu_norm = nn.LayerNorm(embed_dim)

        # ====================================================
        # CROSS MODAL FUSION
        # ====================================================

        self.fusion = CrossModalFusion(

            embed_dim,

            heads,

            dropout,
        )

        # ====================================================
        # TEMPORAL BLOCKS
        # ====================================================

        self.temp_blocks = nn.ModuleList([

            ITBlock(

                embed_dim,

                heads,

                dropout=dropout,

                drop_path=drop_path,
            )

            for _ in range(temp_depth)
        ])

        self.temp_norm = nn.LayerNorm(embed_dim)

        # ====================================================
        # TEMPORAL REFINEMENT
        # ====================================================

        self.temporal_refine = nn.Sequential(

            nn.Conv1d(

                embed_dim,

                embed_dim,

                kernel_size=3,

                padding=1,

                groups=embed_dim,
            ),

            nn.BatchNorm1d(embed_dim),

            nn.GELU(),

            nn.Conv1d(

                embed_dim,

                embed_dim,

                kernel_size=1,
            ),
        )

        # ====================================================
        # ATTENTION POOLING
        # ====================================================

        self.pool = TemporalAttentionPooling(
            embed_dim
        )

        # ====================================================
        # KD PROJECTOR
        # ====================================================

        self.kd_projector = nn.Sequential(

            nn.Linear(
                embed_dim,
                128,
            ),

            nn.LayerNorm(128),
        )

        # ====================================================
        # CLASSIFIER
        # ====================================================

        self.head = nn.Sequential(

            nn.Linear(
                embed_dim,
                embed_dim // 2,
            ),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                embed_dim // 2,
                1,
            ),
        )

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(self, data):

        acc = data["accelerometer"]

        skl = data["skeleton"]

        if (
            skl.ndim == 4
            and
            skl.shape[-1] == 3
        ):

            skl = skl.permute(
                0,
                3,
                1,
                2,
            )

        B, _, T, J = skl.shape

        # ====================================================
        # SKELETON
        # ====================================================

        sf = self.spatial(skl)

        sf = sf.permute(
            0,
            2,
            3,
            1,
        )

        sf = sf.reshape(
            B,
            T,
            J * self.embed_dim,
        )

        sf = self.skel_embed(sf)

        sf = sf + self.pos_skel[:, :T]

        sf = self.pos_drop(sf)

        for blk in self.skel_blocks:

            sf = blk(sf)

        sf = self.skel_norm(sf)

        # ====================================================
        # IMU
        # ====================================================

        imf = acc.permute(0, 2, 1)

        imf = self.imu_conv_stem(imf)

        imf = imf.permute(0, 2, 1)

        imf = imf + self.pos_imu[:, :T]

        imf = self.pos_drop(imf)

        for blk in self.imu_blocks:

            imf = blk(imf)

        imf = self.imu_norm(imf)

        # ====================================================
        # FUSION
        # ====================================================

        fused = self.fusion(
            sf,
            imf,
        )

        # ====================================================
        # TEMPORAL
        # ====================================================

        x = fused + self.pos_temp[:, :T]

        x = self.pos_drop(x)

        for blk in self.temp_blocks:

            x = blk(x)

        x = self.temp_norm(x)

        temp = x.permute(0, 2, 1)

        temp = self.temporal_refine(temp)

        temp = temp.permute(0, 2, 1)

        x = x + temp

        # ====================================================
        # POOLING
        # ====================================================

        pooled = self.pool(x)

        # ====================================================
        # KD FEATURES
        # ====================================================

        kd_features = self.kd_projector(
            pooled
        )

        kd_features = F.normalize(
            kd_features,
            dim=-1,
        )

        # ====================================================
        # CLASSIFICATION
        # ====================================================

        logits = self.head(pooled)

        return logits, kd_features

# ============================================================
# DYNAMIC THRESHOLD
# ============================================================

@torch.no_grad()
def find_best_threshold(model, loader):

    model.eval()

    all_probs = []

    all_labels = []

    for data, labels in loader:

        data = {

            k: v.to(DEVICE)

            for k, v in data.items()
        }

        logits, _ = model(data)

        probs = torch.sigmoid(
            logits.view(-1)
        )

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

    best_acc = 0.0

    best_threshold = 0.5

    thresholds = np.arange(
        0.30,
        0.81,
        0.02,
    )

    for t in thresholds:

        preds = (
            np.array(all_probs) >= t
        ).astype(int)

        acc = accuracy_score(
            all_labels,
            preds,
        )

        if acc > best_acc:

            best_acc = acc

            best_threshold = t

    print(
        f"\nBest Threshold: {best_threshold:.2f}"
    )

    print(
        f"Best Validation Accuracy: {best_acc:.4f}"
    )

    return best_threshold

# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(

    model,

    loader,

    criterion,

    threshold=0.5,
):

    model.eval()

    total_loss = 0.0

    all_probs = []

    all_preds = []

    all_labels = []

    for data, labels in loader:

        data = {

            k: v.to(DEVICE)

            for k, v in data.items()
        }

        labels = labels.float().to(
            DEVICE
        ).view(-1)

        logits, _ = model(data)

        logits = logits.view(-1)

        loss = criterion(
            logits,
            labels,
        )

        probs = torch.sigmoid(logits)

        preds = (
            probs >= threshold
        ).float()

        total_loss += loss.item()

        all_probs.extend(
            probs.detach().cpu().numpy()
        )

        all_preds.extend(
            preds.detach().cpu().numpy()
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(
        all_labels,
        all_preds,
    )

    prec = precision_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    rec = recall_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    f1 = f1_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    auc = roc_auc_score(
        all_labels,
        all_probs,
    )

    return (
        avg_loss,
        acc,
        prec,
        rec,
        f1,
        auc,
    )

# ============================================================
# TRAINING
# ============================================================

def train_teacher(

    model,

    train_loader,

    val_loader,

    test_loader,

    epochs=50,
):

    model.to(DEVICE)

    criterion = FocalLoss(

        alpha=0.80,

        gamma=1.5,
    )

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=1e-4,

        weight_decay=3e-3,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(

        optimizer,

        T_0=5,

        T_mult=2,
    )

    best_acc = 0.0

    best_threshold = 0.5

    patience = 12

    patience_counter = 0

    for epoch in range(epochs):

        model.train()

        pbar = tqdm(

            train_loader,

            desc=f"Teacher Epoch {epoch+1}/{epochs}"
        )

        for data, labels in pbar:

            data = {

                k: v.to(DEVICE)

                for k, v in data.items()
            }

            labels = labels.float().to(
                DEVICE
            ).view(-1)

            # =================================================
            # IMU AUGMENTATION
            # =================================================

            acc = data["accelerometer"]

            noise = (
                torch.randn_like(acc) * 0.01
            )

            scale = 1.0 + (

                torch.randn(

                    acc.size(0),

                    1,

                    1,

                    device=acc.device
                ) * 0.03
            )

            data["accelerometer"] = (
                acc * scale + noise
            )

            optimizer.zero_grad()

            logits, _ = model(data)

            logits = logits.view(-1)

            loss = criterion(
                logits,
                labels,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                1.0,
            )

            optimizer.step()

            pbar.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        scheduler.step()

        threshold = find_best_threshold(

            model,

            val_loader,
        )

        (
            val_loss,
            val_acc,
            val_prec,
            val_rec,
            val_f1,
            val_auc,
        ) = evaluate(

            model,

            val_loader,

            criterion,

            threshold=threshold,
        )

        print("\n" + "="*70)

        print(
            f"Teacher Epoch {epoch+1}/{epochs}"
        )

        print("="*70)

        print(
            f"VAL LOSS   : {val_loss:.4f}"
        )

        print(
            f"VAL ACC    : {val_acc:.4f}"
        )

        print(
            f"VAL PREC   : {val_prec:.4f}"
        )

        print(
            f"VAL REC    : {val_rec:.4f}"
        )

        print(
            f"VAL F1     : {val_f1:.4f}"
        )

        print(
            f"VAL AUC    : {val_auc:.4f}"
        )

        print(
            f"THRESHOLD  : {threshold:.2f}"
        )

        if val_acc > best_acc:

            best_acc = val_acc

            best_threshold = threshold

            patience_counter = 0

            torch.save(

                model.state_dict(),

                "best_teacher_model_v3.pth"
            )

            print(
                "\n✅ BEST TEACHER SAVED"
            )

        else:

            patience_counter += 1

            print(
                f"\nNo improvement "
                f"({patience_counter}/{patience})"
            )

            if patience_counter >= patience:

                print(
                    "\nEARLY STOPPING"
                )

                break

    # ========================================================
    # LOAD BEST
    # ========================================================

    model.load_state_dict(

        torch.load(
            "best_teacher_model_v3.pth"
        )
    )

    # ========================================================
    # FINAL TEST
    # ========================================================

    (
        test_loss,
        test_acc,
        test_prec,
        test_rec,
        test_f1,
        test_auc,
    ) = evaluate(

        model,

        test_loader,

        criterion,

        threshold=best_threshold,
    )

    print("\n" + "="*70)

    print("FINAL TEACHER RESULTS")

    print("="*70)

    print(f"Accuracy  : {test_acc:.4f}")

    print(f"Precision : {test_prec:.4f}")

    print(f"Recall    : {test_rec:.4f}")

    print(f"F1 Score  : {test_f1:.4f}")

    print(f"AUC       : {test_auc:.4f}")

    print(f"Threshold : {best_threshold:.2f}")

    print("="*70)

    return model

# ============================================================
# INITIALIZE TEACHER
# ============================================================

teacher_model = SharedCNNTeacher(

    frames=64,

    joints=32,

    acc_channels=4,

    embed_dim=128,

    heads=8,

    skel_depth=3,

    imu_depth=5,

    temp_depth=2,

    dropout=0.10,

    drop_path=0.10,
)

# ============================================================
# TRAIN TEACHER
# ============================================================

trained_teacher = train_teacher(

    model=teacher_model,

    train_loader=train_loader,

    val_loader=val_loader,

    test_loader=test_loader,

    epochs=50,
)


Using Device: cuda


Teacher Epoch 1/50: 100%|██████████| 214/214 [00:12<00:00, 16.51it/s, loss=0.0558]



Best Threshold: 0.54
Best Validation Accuracy: 0.9586

Teacher Epoch 1/50
VAL LOSS   : 0.0321
VAL ACC    : 0.9586
VAL PREC   : 0.9737
VAL REC    : 0.8706
VAL F1     : 0.9193
VAL AUC    : 0.9890
THRESHOLD  : 0.54

✅ BEST TEACHER SAVED


Teacher Epoch 2/50: 100%|██████████| 214/214 [00:12<00:00, 16.62it/s, loss=0.0124]



Best Threshold: 0.48
Best Validation Accuracy: 0.9575

Teacher Epoch 2/50
VAL LOSS   : 0.0417
VAL ACC    : 0.9575
VAL PREC   : 0.9694
VAL REC    : 0.8706
VAL F1     : 0.9174
VAL AUC    : 0.9849
THRESHOLD  : 0.48

No improvement (1/12)


Teacher Epoch 3/50: 100%|██████████| 214/214 [00:12<00:00, 16.54it/s, loss=0.0069]


... (37 outputs left out)

Teacher Epoch 22/50: 100%|██████████| 214/214 [00:12<00:00, 16.66it/s, loss=0.0000]



Best Threshold: 0.30
Best Validation Accuracy: 0.9660

Teacher Epoch 22/50
VAL LOSS   : 0.0910
VAL ACC    : 0.9660
VAL PREC   : 0.9956
VAL REC    : 0.8784
VAL F1     : 0.9333
VAL AUC    : 0.9829
THRESHOLD  : 0.30

No improvement (11/12)


Teacher Epoch 23/50: 100%|██████████| 214/214 [00:12<00:00, 16.63it/s, loss=0.0002]



Best Threshold: 0.66
Best Validation Accuracy: 0.9671

Teacher Epoch 23/50
VAL LOSS   : 0.0753
VAL ACC    : 0.9671
VAL PREC   : 0.9828
VAL REC    : 0.8941
VAL F1     : 0.9363
VAL AUC    : 0.9746
THRESHOLD  : 0.66

No improvement (12/12)

EARLY STOPPING

FINAL TEACHER RESULTS
Accuracy  : 0.9491
Precision : 0.8526
Recall    : 0.9765
F1 Score  : 0.9103
AUC       : 0.9926
Threshold : 0.36


In [11]:
# ============================================================
# KD SANITY CHECK
# ============================================================

sample_data, sample_labels = next(
    iter(train_loader)
)

sample_data = {

    k: v.to(DEVICE)

    for k, v in sample_data.items()
}

# ------------------------------------------------------------
# TEACHER
# ------------------------------------------------------------

with torch.no_grad():

    teacher_logits, teacher_feat = teacher_model(
        sample_data
    )

# ------------------------------------------------------------
# STUDENT
# ------------------------------------------------------------

student_logits, student_feat, raw_feat = student_model(
    sample_data
)

print("\n" + "="*60)

print("KD SANITY CHECK")

print("="*60)

print(
    "Teacher logits :",
    teacher_logits.shape
)

print(
    "Teacher feat   :",
    teacher_feat.shape
)

print(
    "Student logits :",
    student_logits.shape
)

print(
    "Student feat   :",
    student_feat.shape
)

print(
    "Raw features   :",
    raw_feat.shape
)

print("="*60)


KD SANITY CHECK
Teacher logits : torch.Size([32, 1])
Teacher feat   : torch.Size([32, 128])
Student logits : torch.Size([32, 1])
Student feat   : torch.Size([32, 128])
Raw features   : torch.Size([32, 256])


In [12]:
# ============================================================
# FINAL LIGHTHART KD TRAINING PIPELINE v4
# SMARTFALLMM
# ============================================================
#
# Teacher  : SharedCNNTeacher v3
# Student  : LightHART v4.1
#
# KD LOSSES
# ------------------------------------------------------------
# 1. Stable Focal Hard Loss
# 2. Feature Distillation Loss
# 3. Logit Distillation Loss
#
# ============================================================

import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm

from sklearn.metrics import (

    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# ============================================================
# SEED
# ============================================================

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True

torch.backends.cudnn.benchmark = False

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"\nUsing Device: {DEVICE}")

# ============================================================
# SAVE PATHS
# ============================================================

SAVE_MODEL_PATH = \
"best_lighthart_kd_v4.pth"

SAVE_RESULTS_PATH = \
"kd_results_v4.txt"

# ============================================================
# KD CONFIG
# ============================================================

TEMPERATURE = 3.0

ALPHA_HARD = 0.60

ALPHA_FEAT = 0.30

ALPHA_LOGIT = 0.10

# ============================================================
# STABLE FOCAL LOSS
# ============================================================

class FocalLoss(nn.Module):

    def __init__(

        self,

        alpha=0.80,

        gamma=1.5,
    ):

        super().__init__()

        self.alpha = alpha

        self.gamma = gamma

    def forward(self, logits, targets):

        bce = F.binary_cross_entropy_with_logits(

            logits,

            targets,

            reduction="none",
        )

        probs = torch.sigmoid(logits)

        pt = torch.where(

            targets == 1,

            probs,

            1 - probs,
        )

        focal_weight = (

            1 - pt

        ) ** self.gamma

        loss = self.alpha * focal_weight * bce

        return loss.mean()

# ============================================================
# FEATURE KD LOSS
# ============================================================

def feature_kd_loss(

    student_feat,

    teacher_feat,
):

    if teacher_feat.dim() == 3:

        teacher_feat = teacher_feat.mean(dim=1)

    if student_feat.dim() == 3:

        student_feat = student_feat.mean(dim=1)

    student_feat = F.normalize(

        student_feat,

        dim=-1,
    )

    teacher_feat = F.normalize(

        teacher_feat,

        dim=-1,
    )

    loss = 1 - F.cosine_similarity(

        student_feat,

        teacher_feat,

        dim=-1,
    )

    return loss.mean()

# ============================================================
# LOGIT KD LOSS
# ============================================================

def logit_kd_loss(

    student_logits,

    teacher_logits,

    temperature,
):

    student_probs = torch.sigmoid(
        student_logits / temperature
    )

    teacher_probs = torch.sigmoid(
        teacher_logits / temperature
    )

    loss = F.mse_loss(

        student_probs,

        teacher_probs,
    )

    return loss * (temperature ** 2)

# ============================================================
# DYNAMIC THRESHOLD
# ============================================================

@torch.no_grad()
def find_best_threshold(

    model,

    loader,
):

    model.eval()

    all_probs = []

    all_labels = []

    for data, labels in loader:

        data = {

            k: v.to(DEVICE)

            for k, v in data.items()
        }

        logits, _, _ = model(data)

        probs = torch.sigmoid(
            logits.view(-1)
        )

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )

    best_acc = 0.0

    best_threshold = 0.5

    thresholds = np.arange(
        0.30,
        0.81,
        0.02,
    )

    for t in thresholds:

        preds = (
            np.array(all_probs) >= t
        ).astype(int)

        acc = accuracy_score(
            all_labels,
            preds,
        )

        if acc > best_acc:

            best_acc = acc

            best_threshold = t

    print(
        f"\nBest Threshold: {best_threshold:.2f}"
    )

    print(
        f"Best Validation Accuracy: {best_acc:.4f}"
    )

    return best_threshold

# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_kd(

    model,

    loader,

    criterion,

    threshold=0.5,
):

    model.eval()

    total_loss = 0.0

    all_probs = []

    all_preds = []

    all_labels = []

    for data, labels in loader:

        data = {

            k: v.to(DEVICE)

            for k, v in data.items()
        }

        labels = labels.float().to(
            DEVICE
        ).view(-1)

        logits, _, _ = model(data)

        logits = logits.view(-1)

        loss = criterion(
            logits,
            labels,
        )

        probs = torch.sigmoid(logits)

        preds = (
            probs >= threshold
        ).float()

        total_loss += loss.item()

        all_probs.extend(
            probs.detach().cpu().numpy()
        )

        all_preds.extend(
            preds.detach().cpu().numpy()
        )

        all_labels.extend(
            labels.detach().cpu().numpy()
        )

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(
        all_labels,
        all_preds,
    )

    prec = precision_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    rec = recall_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    f1 = f1_score(
        all_labels,
        all_preds,
        zero_division=0,
    )

    auc = roc_auc_score(
        all_labels,
        all_probs,
    )

    return (

        avg_loss,

        acc,

        prec,

        rec,

        f1,

        auc,
    )

# ============================================================
# KD TRAINING
# ============================================================

def train_kd(

    teacher,

    student,

    train_loader,

    val_loader,

    test_loader,

    epochs=35,
):

    teacher.to(DEVICE)

    student.to(DEVICE)

    # ========================================================
    # FREEZE TEACHER
    # ========================================================

    teacher.eval()

    for param in teacher.parameters():

        param.requires_grad = False

    # ========================================================
    # LOSS
    # ========================================================

    hard_criterion = FocalLoss(

        alpha=0.80,

        gamma=1.5,
    )

    # ========================================================
    # OPTIMIZER
    # ========================================================

    optimizer = torch.optim.AdamW(

        student.parameters(),

        lr=1e-4,

        weight_decay=3e-3,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(

        optimizer,

        T_0=5,

        T_mult=2,
    )

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    best_f1 = 0.0

    best_threshold = 0.5

    patience = 12

    patience_counter = 0

    # ========================================================
    # TRAIN LOOP
    # ========================================================

    for epoch in range(epochs):

        student.train()

        running_total = 0.0

        running_hard = 0.0

        running_feat = 0.0

        running_logit = 0.0

        pbar = tqdm(

            train_loader,

            desc=f"KD Epoch {epoch+1}/{epochs}"
        )

        for data, labels in pbar:

            data = {

                k: v.to(DEVICE)

                for k, v in data.items()
            }

            labels = labels.float().to(
                DEVICE
            ).view(-1)

            # =================================================
            # AUGMENTATION
            # =================================================

            if "accelerometer" in data:

                acc = data["accelerometer"]

                noise = (
                    torch.randn_like(acc) * 0.015
                )

                scale = 1.0 + (

                    torch.randn(

                        acc.size(0),

                        1,

                        1,

                        device=acc.device
                    ) * 0.05
                )

                data["accelerometer"] = (
                    acc * scale + noise
                )

            optimizer.zero_grad()

            # =================================================
            # TEACHER
            # =================================================

            with torch.no_grad():

                teacher_logits, teacher_feat = teacher(
                    data
                )

                teacher_logits = teacher_logits.view(-1)

            # =================================================
            # STUDENT
            # =================================================

            student_logits, student_feat, _ = student(
                data
            )

            student_logits = student_logits.view(-1)

            # =================================================
            # HARD LOSS
            # =================================================

            hard_loss = hard_criterion(

                student_logits,

                labels,
            )

            # =================================================
            # FEATURE KD LOSS
            # =================================================

            feat_loss = feature_kd_loss(

                student_feat,

                teacher_feat,
            )

            # =================================================
            # LOGIT KD LOSS
            # =================================================

            kd_loss = logit_kd_loss(

                student_logits,

                teacher_logits,

                TEMPERATURE,
            )

            # =================================================
            # TOTAL LOSS
            # =================================================

            total_loss = (

                ALPHA_HARD * hard_loss

                +

                ALPHA_FEAT * feat_loss

                +

                ALPHA_LOGIT * kd_loss
            )

            # =================================================
            # BACKPROP
            # =================================================

            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(

                student.parameters(),

                1.0,
            )

            optimizer.step()

            # =================================================
            # LOGGING
            # =================================================

            running_total += total_loss.item()

            running_hard += hard_loss.item()

            running_feat += feat_loss.item()

            running_logit += kd_loss.item()

            pbar.set_postfix(

                total=f"{total_loss.item():.4f}",

                hard=f"{hard_loss.item():.4f}",
            )

        scheduler.step()

        # ====================================================
        # DYNAMIC THRESHOLD
        # ====================================================

        threshold = find_best_threshold(

            student,

            val_loader,
        )

        # ====================================================
        # VALIDATION
        # ====================================================

        (
            val_loss,
            val_acc,
            val_prec,
            val_rec,
            val_f1,
            val_auc,
        ) = evaluate_kd(

            student,

            val_loader,

            hard_criterion,

            threshold=threshold,
        )

        print("\n" + "="*70)

        print(
            f"KD Epoch {epoch+1}/{epochs}"
        )

        print("="*70)

        print(
            f"TOTAL LOSS : "
            f"{running_total / len(train_loader):.4f}"
        )

        print(
            f"HARD LOSS  : "
            f"{running_hard / len(train_loader):.4f}"
        )

        print(
            f"FEAT LOSS  : "
            f"{running_feat / len(train_loader):.4f}"
        )

        print(
            f"LOGIT LOSS : "
            f"{running_logit / len(train_loader):.4f}"
        )

        print("-"*70)

        print(
            f"VAL LOSS : {val_loss:.4f}"
        )

        print(
            f"VAL ACC  : {val_acc:.4f}"
        )

        print(
            f"VAL PREC : {val_prec:.4f}"
        )

        print(
            f"VAL REC  : {val_rec:.4f}"
        )

        print(
            f"VAL F1   : {val_f1:.4f}"
        )

        print(
            f"VAL AUC  : {val_auc:.4f}"
        )

        print(
            f"THRESHOLD: {threshold:.2f}"
        )

        # ====================================================
        # SAVE BEST
        # ====================================================

        if val_f1 > best_f1:

            best_f1 = val_f1

            best_threshold = threshold

            patience_counter = 0

            torch.save(

                student.state_dict(),

                SAVE_MODEL_PATH
            )

            print(
                "\n🌟 BEST KD MODEL SAVED"
            )

        else:

            patience_counter += 1

            print(

                f"\nNo improvement "

                f"({patience_counter}/{patience})"
            )

            if patience_counter >= patience:

                print(
                    "\n🛑 EARLY STOPPING"
                )

                break

    # ========================================================
    # LOAD BEST
    # ========================================================

    print("\nLoading best KD student...")

    student.load_state_dict(

        torch.load(
            SAVE_MODEL_PATH
        )
    )

    # ========================================================
    # FINAL TEST
    # ========================================================

    (
        test_loss,
        test_acc,
        test_prec,
        test_rec,
        test_f1,
        test_auc,
    ) = evaluate_kd(

        student,

        test_loader,

        hard_criterion,

        threshold=best_threshold,
    )

    print("\n" + "="*70)

    print("FINAL KD RESULTS")

    print("="*70)

    print(f"Loss      : {test_loss:.4f}")

    print(f"Accuracy  : {test_acc:.4f}")

    print(f"Precision : {test_prec:.4f}")

    print(f"Recall    : {test_rec:.4f}")

    print(f"F1 Score  : {test_f1:.4f}")

    print(f"AUC       : {test_auc:.4f}")

    print(f"Threshold : {best_threshold:.2f}")

    print("="*70)

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    with open(SAVE_RESULTS_PATH, "w") as f:

        f.write(
            "==================================================\n"
        )

        f.write(
            "FINAL LIGHTHART KD RESULTS\n"
        )

        f.write(
            "==================================================\n\n"
        )

        f.write(
            f"Loss      : {test_loss:.4f}\n"
        )

        f.write(
            f"Accuracy  : {test_acc:.4f}\n"
        )

        f.write(
            f"Precision : {test_prec:.4f}\n"
        )

        f.write(
            f"Recall    : {test_rec:.4f}\n"
        )

        f.write(
            f"F1 Score  : {test_f1:.4f}\n"
        )

        f.write(
            f"AUC       : {test_auc:.4f}\n"
        )

        f.write(
            f"Threshold : {best_threshold:.2f}\n"
        )

    print(
        f"\n✅ KD Results saved to:\n"
        f"{SAVE_RESULTS_PATH}"
    )

    return student

# ============================================================
# LOAD FINAL TEACHER
# ============================================================

teacher_model = SharedCNNTeacher(

    frames=64,

    joints=32,

    acc_channels=4,

    embed_dim=128,

    heads=8,

    skel_depth=3,

    imu_depth=5,

    temp_depth=2,

    dropout=0.10,

    drop_path=0.10,
)

teacher_model.load_state_dict(

    torch.load(
        "best_teacher_model_v3.pth"
    )
)

print("\n✅ Teacher loaded successfully.")

# ============================================================
# CREATE FINAL STUDENT
# ============================================================

student_model = LightHART(

    embed_dim=128,

    num_heads=8,

    num_layers=5,

    dropout=0.08,
)

print("✅ LightHART v4.1 student initialized.")

# ============================================================
# TRAIN KD
# ============================================================

kd_student = train_kd(

    teacher=teacher_model,

    student=student_model,

    train_loader=train_loader,

    val_loader=val_loader,

    test_loader=test_loader,

    epochs=35,
)


Using Device: cuda

✅ Teacher loaded successfully.
✅ LightHART v4.1 student initialized.


KD Epoch 1/35: 100%|██████████| 214/214 [00:07<00:00, 29.84it/s, hard=0.1255, total=0.2249]



Best Threshold: 0.58
Best Validation Accuracy: 0.7495

KD Epoch 1/35
TOTAL LOSS : 0.2549
HARD LOSS  : 0.1342
FEAT LOSS  : 0.4062
LOGIT LOSS : 0.5258
----------------------------------------------------------------------
VAL LOSS : 0.1606
VAL ACC  : 0.7495
VAL PREC : 0.5785
VAL REC  : 0.2745
VAL F1   : 0.3723
VAL AUC  : 0.7998
THRESHOLD: 0.58

🌟 BEST KD MODEL SAVED


KD Epoch 2/35: 100%|██████████| 214/214 [00:06<00:00, 30.92it/s, hard=0.1180, total=0.1901]



Best Threshold: 0.48
Best Validation Accuracy: 0.7856

KD Epoch 2/35
TOTAL LOSS : 0.1932
HARD LOSS  : 0.1141
FEAT LOSS  : 0.2735
LOGIT LOSS : 0.4265
----------------------------------------------------------------------
VAL LOSS : 0.1459
VAL ACC  : 0.7856
VAL PREC : 0.6710
VAL REC  : 0.4078
VAL F1   : 0.5073
VAL AUC  : 0.8252
THRESHOLD: 0.48

🌟 BEST KD MODEL SAVED


KD Epoch 3/35: 100%|██████████| 214/214 [00:06<00:00, 30.94it/s, hard=0.0659, total=0.1416]


... (29 outputs left out)

KD Epoch 18/35: 100%|██████████| 214/214 [00:07<00:00, 29.82it/s, hard=0.0494, total=0.0827]



Best Threshold: 0.62
Best Validation Accuracy: 0.7983

KD Epoch 18/35
TOTAL LOSS : 0.0959
HARD LOSS  : 0.0590
FEAT LOSS  : 0.1428
LOGIT LOSS : 0.1773
----------------------------------------------------------------------
VAL LOSS : 0.2523
VAL ACC  : 0.7983
VAL PREC : 0.7778
VAL REC  : 0.3569
VAL F1   : 0.4892
VAL AUC  : 0.8387
THRESHOLD: 0.62

No improvement (11/12)


KD Epoch 19/35: 100%|██████████| 214/214 [00:06<00:00, 30.87it/s, hard=0.0298, total=0.0637]



Best Threshold: 0.62
Best Validation Accuracy: 0.7792

KD Epoch 19/35
TOTAL LOSS : 0.0880
HARD LOSS  : 0.0532
FEAT LOSS  : 0.1332
LOGIT LOSS : 0.1607
----------------------------------------------------------------------
VAL LOSS : 0.2379
VAL ACC  : 0.7792
VAL PREC : 0.7009
VAL REC  : 0.3216
VAL F1   : 0.4409
VAL AUC  : 0.8204
THRESHOLD: 0.62

No improvement (12/12)

🛑 EARLY STOPPING

Loading best KD student...

FINAL KD RESULTS
Loss      : 0.1045
Accuracy  : 0.8729
Precision : 0.7082
Recall    : 0.8840
F1 Score  : 0.7864
AUC       : 0.9493
Threshold : 0.50

✅ KD Results saved to:
kd_results_v4.txt


In [13]:
import sys

!{sys.executable} -m pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 12.4 MB/s eta 0:00:00


In [14]:
import sys

!{sys.executable} -m pip uninstall -y ml_dtypes
!{sys.executable} -m pip install ml_dtypes==0.4.1
!{sys.executable} -m pip install --upgrade onnx onnxscript onnx_ir

Found existing installation: ml_dtypes 0.5.4
Uninstalling ml_dtypes-0.5.4:
  Successfully uninstalled ml_dtypes-0.5.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 36.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnx-ir 0.2.1 requires ml_dtypes>=0.5.0, but you have ml-dtypes 0.4.1 which is incompatible.
onnx 1.20.1 requires ml_dtypes>=0.5.0, but you have ml-dtypes 0.4.1 which is incompatible.
jaxlib 0.7.2 requires ml_dtypes>=0.5.0, but you have ml-dtypes 0.4.1 which is incompatible.
jax 0.7.2 requires ml_dtypes>=0.5.0, but you have ml-dtypes 0.4.1 which is incompatible.
tensorflow 2.19.0 requires ml-dtypes<1.0.0,>=0.5.1, but you have ml-dtypes 0.4.1 which is incompatible.
tensorstore 0.1.81 requires ml_dtypes>=0.5.0, but you have ml-dtypes 0.4.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 93.0 M

In [15]:
import tensorflow as tf

import onnx

import tensorflow_addons as tfa

from onnx_tf.backend import prepare

print(tf.__version__)

print("✅ EVERYTHING WORKS")

2026-05-20 14:18:13.742554: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779286693.981862      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779286694.046197      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779286694.606522      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779286694.606563      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779286694.606566      23 computation_placer.cc:177] computation placer alr

ModuleNotFoundError: No module named 'tensorflow_addons'

In [ ]:
import sys

!{sys.executable} -m pip install torch torchvision torchaudio

In [ ]:
# ============================================================
# LIGHTHART-KD → TFLITE PIPELINE
# FINAL WORKING VERSION
# ============================================================

# ============================================================
# FORCE CPU
# ============================================================

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# ============================================================
# IMPORTS
# ============================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F

import tensorflow as tf
import onnx

from onnx_tf.backend import prepare

import numpy as np

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device("cpu")

print(f"\nUsing Device: {DEVICE}")

# ============================================================
# TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(

        self,

        embed_dim,

        num_heads,

        dropout=0.2,
    ):

        super().__init__()

        # ----------------------------------------------------
        # NORM 1
        # ----------------------------------------------------

        self.norm1 = nn.LayerNorm(
            embed_dim
        )

        # ----------------------------------------------------
        # ATTENTION
        # ----------------------------------------------------

        self.attn = nn.MultiheadAttention(

            embed_dim=embed_dim,

            num_heads=num_heads,

            dropout=dropout,

            batch_first=True,
        )

        # ----------------------------------------------------
        # NORM 2
        # ----------------------------------------------------

        self.norm2 = nn.LayerNorm(
            embed_dim
        )

        # ----------------------------------------------------
        # FFN
        # ----------------------------------------------------

        self.ffn = nn.Sequential(

            nn.Linear(
                embed_dim,
                embed_dim * 2
            ),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(
                embed_dim * 2,
                embed_dim
            ),

            nn.Dropout(dropout),
        )

    def forward(self, x):

        # ----------------------------------------------------
        # ATTENTION
        # ----------------------------------------------------

        attn_input = self.norm1(x)

        attn_output, _ = self.attn(

            attn_input,

            attn_input,

            attn_input,
        )

        x = x + attn_output

        # ----------------------------------------------------
        # FFN
        # ----------------------------------------------------

        ffn_input = self.norm2(x)

        x = x + self.ffn(ffn_input)

        return x

# ============================================================
# EXACT LIGHTHART ARCHITECTURE
# ============================================================

class LightHART(nn.Module):

    def __init__(

        self,

        acc_frames=64,

        acc_coords=4,

        embed_dim=64,

        num_heads=4,

        num_layers=3,

        dropout=0.2,
    ):

        super().__init__()

        # ----------------------------------------------------
        # CONV PROJECTION
        # ----------------------------------------------------

        self.conv_proj = nn.Conv1d(

            in_channels=acc_coords,

            out_channels=embed_dim,

            kernel_size=8,

            padding=4,

            bias=False,
        )

        # ----------------------------------------------------
        # BATCH NORM
        # ----------------------------------------------------

        self.bn_proj = nn.BatchNorm1d(
            embed_dim
        )

        # ----------------------------------------------------
        # TRANSFORMER BLOCKS
        # ----------------------------------------------------

        self.encoder_blocks = nn.ModuleList([

            TransformerBlock(

                embed_dim=embed_dim,

                num_heads=num_heads,

                dropout=dropout,
            )

            for _ in range(num_layers)
        ])

        # ----------------------------------------------------
        # TEMPORAL NORM
        # ----------------------------------------------------

        self.temporal_norm = nn.LayerNorm(
            embed_dim
        )

        # ----------------------------------------------------
        # CLASSIFIER
        # ----------------------------------------------------

        self.classifier = nn.Linear(

            embed_dim,

            1
        )

        # ----------------------------------------------------
        # KD PROJECTOR
        # ONLY FOR CHECKPOINT MATCHING
        # ----------------------------------------------------

        self.kd_projector = nn.Sequential(

            nn.Linear(
                embed_dim,
                embed_dim
            ),

            nn.BatchNorm1d(

                embed_dim,

                track_running_stats=False
            ),
        )

    def forward(self, x):

        # ----------------------------------------------------
        # INPUT
        # ----------------------------------------------------

        x = x.permute(0, 2, 1)

        # ----------------------------------------------------
        # CONV
        # ----------------------------------------------------

        x = self.conv_proj(x)

        x = self.bn_proj(x)

        x = x.permute(0, 2, 1)

        # ----------------------------------------------------
        # TRANSFORMER
        # ----------------------------------------------------

        for block in self.encoder_blocks:

            x = block(x)

        # ----------------------------------------------------
        # TEMPORAL NORM
        # ----------------------------------------------------

        x = self.temporal_norm(x)

        # ----------------------------------------------------
        # GLOBAL POOLING
        # ----------------------------------------------------

        pooled = x.mean(dim=1)

        # ----------------------------------------------------
        # CLASSIFICATION
        # ----------------------------------------------------

        logits = self.classifier(
            pooled
        )

        # ----------------------------------------------------
        # IMPORTANT FIX
        # DO NOT USE KD PROJECTOR
        # DURING DEPLOYMENT
        # ----------------------------------------------------

        kd_features = pooled

        return logits, kd_features, pooled

# ============================================================
# CREATE MODEL
# ============================================================

model = LightHART(

    acc_coords=4,

    embed_dim=64,

    num_heads=4,

    num_layers=3,

    dropout=0.2,
)

# ============================================================
# LOAD TRAINED WEIGHTS
# ============================================================

state_dict = torch.load(

    r"C:/Users/praty/Documents/best_lighthart_kd.pth",

    map_location=torch.device("cpu"),

    weights_only=True,
)

model.load_state_dict(
    state_dict
)

# ============================================================
# FORCE ALL BN TO EVAL
# ============================================================

model.eval()

for module in model.modules():

    if isinstance(module, nn.BatchNorm1d):

        module.eval()

print("\n✅ LIGHTHART-KD WEIGHTS LOADED")

# ============================================================
# DEPLOYMENT WRAPPER
# ============================================================

class DeployModel(nn.Module):

    def __init__(self, model):

        super().__init__()

        self.model = model

    def forward(self, x):

        logits, _, _ = self.model(x)

        return logits

deploy_model = DeployModel(
    model
)

deploy_model.eval()

print("\n✅ DEPLOYMENT MODEL READY")

# ============================================================
# DUMMY INPUT
# ============================================================

dummy_input = torch.randn(

    1,

    64,

    4
)

# ============================================================
# TEST FORWARD PASS
# ============================================================

with torch.no_grad():

    output = deploy_model(dummy_input)

print("\n✅ FORWARD PASS SUCCESSFUL")

print("\nOutput Shape:")

print(output.shape)

# ============================================================
# EXPORT ONNX
# ============================================================

ONNX_PATH = \
r"C:/Users/praty/Documents/lighthart_kd.onnx"

torch.onnx.export(

    deploy_model,

    dummy_input,

    ONNX_PATH,

    export_params=True,

    opset_version=13,

    do_constant_folding=True,

    input_names=["accelerometer"],

    output_names=["output"],

    dynamic_axes={

        "accelerometer": {0: "batch_size"},

        "output": {0: "batch_size"},
    }
)

print("\n✅ ONNX MODEL EXPORTED")

print(f"\nSaved To:\n{ONNX_PATH}")

# ============================================================
# VERIFY ONNX
# ============================================================

onnx_model = onnx.load(
    ONNX_PATH
)

onnx.checker.check_model(
    onnx_model
)

print("\n✅ ONNX MODEL VERIFIED")

# ============================================================
# ONNX → TENSORFLOW
# ============================================================

tf_rep = prepare(
    onnx_model
)

TF_PATH = \
r"C:/Users/praty/Documents/lighthart_tf"

tf_rep.export_graph(
    TF_PATH
)

print("\n✅ TENSORFLOW MODEL SAVED")

print(f"\nSaved To:\n{TF_PATH}")

# ============================================================
# TF → TFLITE
# ============================================================

converter = tf.lite.TFLiteConverter.from_saved_model(
    TF_PATH
)

# ------------------------------------------------------------
# OPTIMIZATION
# ------------------------------------------------------------

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

# ============================================================
# CONVERT
# ============================================================

tflite_model = converter.convert()

# ============================================================
# SAVE TFLITE
# ============================================================

TFLITE_PATH = \
r"C:/Users/praty/Documents/lighthart_kd.tflite"

with open(TFLITE_PATH, "wb") as f:

    f.write(tflite_model)

print("\n✅ TFLITE MODEL SAVED")

print(f"\nSaved To:\n{TFLITE_PATH}")

# ============================================================
# TFLITE MODEL SIZE
# ============================================================

tflite_size_mb = os.path.getsize(

    TFLITE_PATH

) / (1024 * 1024)

print("\n================================================")

print(f"TFLITE MODEL SIZE: {tflite_size_mb:.2f} MB")

print("================================================")

# ============================================================
# TEST TFLITE
# ============================================================

interpreter = tf.lite.Interpreter(

    model_path=TFLITE_PATH
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()

output_details = interpreter.get_output_details()

print("\n✅ TFLITE MODEL LOADED")

# ============================================================
# TEST INPUT
# ============================================================

test_input = np.random.randn(

    1,

    64,

    4
).astype(np.float32)

interpreter.set_tensor(

    input_details[0]["index"],

    test_input
)

interpreter.invoke()

output = interpreter.get_tensor(

    output_details[0]["index"]
)

print("\n✅ TFLITE INFERENCE SUCCESSFUL")

print("\nTFLITE OUTPUT:")

print(output)

print("\n🎉 COMPLETE PIPELINE SUCCESSFUL")

In [ ]:
import os

save_dir = "processed_data"
os.makedirs(save_dir, exist_ok=True)

print("Saving arrays to disk...")
np.save(f"{save_dir}/acc_set.npy", dataset["accelerometer"])
np.save(f"{save_dir}/skl_set.npy", dataset["skeleton"])
np.save(f"{save_dir}/labels.npy", dataset["labels"])
np.save(f"{save_dir}/subjects.npy", dataset["subjects"])
np.save(f"{save_dir}/activities.npy", dataset["activities"])
print("Done! You can now load these instantly for training.")


In [ ]:
from collections import Counter

subject_counts = Counter(dataset["subjects"])

for s in sorted(subject_counts):
    print(f"Subject {s}: {subject_counts[s]} samples")